# Painel de SST e Conformidade · versão 2 em Python

<a href="https://colab.research.google.com/github/larissahorrara/painel-sst-conformidade/blob/main/python/painel_sst_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"></a>

Este notebook reconstrói em Python o **Painel de Saúde, Segurança do Trabalho e Conformidade** do repositório [painel-sst-conformidade](https://github.com/larissahorrara/painel-sst-conformidade). Ele gera os dados fictícios, calcula os indicadores com pandas e NumPy, monta os gráficos com Plotly e exporta um painel em HTML com cinco abas:

1. **Visão Geral**: Taxa de Frequência, Taxa de Gravidade, Conformidade NR-1, Satisfação em SST e alertas.
2. **Riscos e Brigada**: matriz de risco 5×5 da NR-1, brigada de incêndio e validade dos equipamentos.
3. **Governança**: vigência dos documentos legais e fluxo de revisão dos POPs e do PAE.
4. **Percepção**: pesquisa de clima de segurança respondida pelos colaboradores.
5. **Selos e Certificações**: impacto dos indicadores em GPTW, ISO 45001 e ESG, com a Tabela de Alerta Vermelho.

**Como executar:** no menu, escolha **Ambiente de execução › Executar tudo**. Ao final, o Colab baixa o painel (`painel_sst.html`) e a base de dados (`base_sst.xlsx`). Fora do Colab, os arquivos ficam na mesma pasta do notebook.

> Todos os nomes, setores e números são **fictícios** e foram gerados para demonstração. Nenhum dado se refere a pessoas ou empresas reais.

## Etapa 1 · Preparar o ambiente

Importa as bibliotecas usadas no notebook. No Google Colab, pandas, NumPy, Plotly e openpyxl já vêm instalados. A célula também identifica se o notebook está rodando no Colab, porque o download dos arquivos ao final só existe lá.

In [ ]:
import html
import math
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import get_plotlyjs
from IPython.display import HTML, display

try:
    from google.colab import files as arquivos_colab
    NO_COLAB = True
except ImportError:
    arquivos_colab = None
    NO_COLAB = False

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

print(f'pandas {pd.__version__} · NumPy {np.__version__} · Plotly {plotly.__version__}')
print('Executando no Google Colab' if NO_COLAB else 'Executando fora do Colab (Jupyter ou VS Code)')

## Etapa 2 · Definir os parâmetros

Todos os parâmetros do painel ficam aqui: data de referência, setores e efetivo, prédios, turnos, metas e a **semente** dos números aleatórios.

- Troque o valor de `SEMENTE` para gerar outro cenário fictício.
- Altere o dicionário `METAS` para testar outras metas. Metas de taxa (`tf` e `tg`) são valores máximos; as demais são percentuais mínimos.

In [ ]:
SEMENTE = 2026
DATA_REFERENCIA = pd.Timestamp('2026-10-16')
EMPRESA = 'Empresa Exemplo S.A.'

# Setores e número de colaboradores em cada um
SETORES = {
    'Administrativo': 90,
    'Produção': 180,
    'Manutenção': 80,
    'Logística': 110,
    'Laboratório': 60,
    'Atendimento': 80,
}
PREDIOS = ['Bloco A', 'Bloco B', 'Bloco C']
TURNOS = ['Manhã', 'Tarde', 'Noite']
EFETIVO = sum(SETORES.values())

MESES_HISTORICO = 24          # 12 meses analisados + 12 meses para comparação
BRIGADISTAS_MINIMOS = 3       # por prédio e turno
DIAS_FLUXO_PARADO = 60        # POP ou PAE parado na mesma etapa

CATEGORIAS_RISCO = ['Físico', 'Químico', 'Biológico', 'Ergonômico', 'Acidentes']
DIMENSOES = [
    'Eficácia dos treinamentos',
    'Liderança – saúde física',
    'Comunicação sobre riscos',
    'Confiança para relatar incidentes',
    'Direito de recusa',
    'Liderança – saúde mental',
]
CICLOS = ['4º tri/2025', '1º tri/2026', '2º tri/2026', '3º tri/2026']

METAS = {
    'tf': 12,                     # Taxa de Frequência máxima
    'tg': 400,                    # Taxa de Gravidade máxima
    'conformidade_nr1': 90,       # % de riscos com plano de ação
    'brigada_valida': 95,         # % de brigadistas com reciclagem válida
    'equipamentos_validos': 95,   # % de equipamentos dentro da validade
    'documentos_vigentes': 100,   # % de documentos legais vigentes
    'participacao': 75,           # % de colaboradores que responderam a pesquisa
    'favoravel': 70,              # % de respostas favoráveis (notas 4 e 5)
    'aso_em_dia': 95,             # % de ASOs periódicos em dia
    'pops_vigentes': 85,          # % de POPs e PAE vigentes
}

rng = np.random.default_rng(SEMENTE)

MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']


def rotulo_mes(periodo):
    """Converte um período mensal em rótulo curto, como 'Nov/25'."""
    return f'{MESES_PT[periodo.month - 1]}/{str(periodo.year)[2:]}'


def faixa_prazo(dias):
    """Classifica um prazo em dias. Vale para ASO, brigada, equipamentos e documentos."""
    if dias < 0:
        return 'Vencido'
    if dias <= 30:
        return 'Vence em até 30 dias'
    if dias <= 60:
        return 'Vence em 31 a 60 dias'
    return 'Em dia'


FAIXAS = ['Vencido', 'Vence em até 30 dias', 'Vence em 31 a 60 dias', 'Em dia']

MESES = pd.period_range(end=DATA_REFERENCIA.to_period('M'), periods=MESES_HISTORICO, freq='M')
MESES_ATUAIS = MESES[-12:]
MESES_ANTERIORES = MESES[:-12]
PERIODO_TEXTO = f'{rotulo_mes(MESES_ATUAIS[0])} a {rotulo_mes(MESES_ATUAIS[-1])}'

print(f'{EMPRESA} · {EFETIVO} colaboradores · período analisado: {PERIODO_TEXTO}')

## Etapa 3 · Gerar os colaboradores e o vencimento dos ASOs

Cria a base de colaboradores, com setor, prédio, turno e data de admissão. Para cada pessoa, registra o último ASO periódico (NR-7) e calcula o vencimento, considerando periodicidade anual.

In [ ]:
PESO_TURNO = {
    'Administrativo': [0.85, 0.15, 0.00],
    'Atendimento': [0.50, 0.40, 0.10],
}
PESO_PREDIO = {
    'Administrativo': [0.20, 0.60, 0.20],
    'Laboratório': [0.15, 0.15, 0.70],
    'Produção': [0.50, 0.25, 0.25],
}

linhas = []
for setor, quantidade in SETORES.items():
    for _ in range(quantidade):
        admissao = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(60, 365 * 15)))
        ultimo_aso = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(0, 400)))
        ultimo_aso = max(ultimo_aso, admissao)
        linhas.append({
            'setor': setor,
            'predio': rng.choice(PREDIOS, p=PESO_PREDIO.get(setor, [1 / 3] * 3)),
            'turno': rng.choice(TURNOS, p=PESO_TURNO.get(setor, [0.40, 0.35, 0.25])),
            'data_admissao': admissao,
            'ultimo_aso': ultimo_aso,
        })

colaboradores = pd.DataFrame(linhas)
colaboradores.insert(0, 'colaborador_id', [f'C{i:04d}' for i in range(1, len(colaboradores) + 1)])
colaboradores['vencimento_aso'] = colaboradores['ultimo_aso'] + pd.Timedelta(days=365)
colaboradores['dias_para_vencer_aso'] = (colaboradores['vencimento_aso'] - DATA_REFERENCIA).dt.days
colaboradores['situacao_aso'] = colaboradores['dias_para_vencer_aso'].apply(faixa_prazo)

aso_em_dia = 100 * (colaboradores['situacao_aso'] != 'Vencido').mean()
print(f'{len(colaboradores)} colaboradores · ASO em dia: {aso_em_dia:.1f}%')
colaboradores.head()

## Etapa 4 · Gerar horas trabalhadas e acidentes e calcular TF e TG

Gera 24 meses de horas-homem trabalhadas (HHT) e de acidentes por setor. As taxas seguem a **ABNT NBR 14280**:

- **Taxa de Frequência (TF)** = acidentes com afastamento × 1.000.000 ÷ HHT
- **Taxa de Gravidade (TG)** = (dias perdidos + dias debitados) × 1.000.000 ÷ HHT

O painel mostra o **acumulado móvel de 12 meses**, comparado aos 12 meses anteriores.

In [ ]:
# Acidentes com afastamento por milhão de HHT, por setor (12 meses mais recentes)
TAXA_BASE = {
    'Administrativo': 2.5,
    'Produção': 10.0,
    'Manutenção': 24.0,
    'Logística': 9.0,
    'Laboratório': 6.0,
    'Atendimento': 3.5,
}
FATOR_ANO_ANTERIOR = 1.25     # o ano anterior teve mais acidentes
NATUREZAS = {
    'Administrativo': ['Queda de mesmo nível', 'Esforço excessivo', 'Batida contra objeto'],
    'Produção': ['Corte', 'Prensagem de membro', 'Esforço excessivo', 'Projeção de partícula'],
    'Manutenção': ['Choque elétrico', 'Queda de altura', 'Corte', 'Queimadura'],
    'Logística': ['Atropelamento por empilhadeira', 'Esforço excessivo', 'Queda de material'],
    'Laboratório': ['Contato com produto químico', 'Corte com vidraria', 'Queimadura'],
    'Atendimento': ['Queda de mesmo nível', 'Esforço excessivo', 'Agressão verbal com afastamento'],
}
PARTES_CORPO = ['Mãos e dedos', 'Membros inferiores', 'Coluna', 'Olhos', 'Membros superiores', 'Cabeça']

linhas_horas, linhas_acidentes = [], []
for mes in MESES:
    fim = min(mes.end_time.normalize(), DATA_REFERENCIA) + pd.Timedelta(days=1)   # o mês atual vai até a data de referência
    dias_uteis = np.busday_count(mes.start_time.date(), fim.date())
    fator = 1.0 if mes in MESES_ATUAIS else FATOR_ANO_ANTERIOR
    for setor, quantidade in SETORES.items():
        hht = quantidade * dias_uteis * 8 * rng.uniform(0.95, 1.03)
        linhas_horas.append({'mes': str(mes), 'setor': setor, 'colaboradores': quantidade, 'hht': round(hht)})

        esperado = hht * TAXA_BASE[setor] * fator / 1e6
        for tipo, media in [('Com afastamento', esperado), ('Sem afastamento', esperado * 1.8)]:
            for _ in range(rng.poisson(media)):
                data = mes.start_time + pd.Timedelta(days=int(rng.integers(0, (fim - mes.start_time).days)))
                perdidos = int(min(90, rng.gamma(1.5, 10) + 1)) if tipo == 'Com afastamento' else 0
                debitados = int(rng.choice([100, 150, 200, 300])) if tipo == 'Com afastamento' and rng.random() < 0.04 else 0
                linhas_acidentes.append({
                    'data': data, 'mes': str(mes), 'setor': setor, 'tipo': tipo,
                    'natureza': rng.choice(NATUREZAS[setor]), 'parte_do_corpo': rng.choice(PARTES_CORPO),
                    'dias_perdidos': perdidos, 'dias_debitados': debitados,
                })

horas_trabalhadas = pd.DataFrame(linhas_horas)
acidentes = pd.DataFrame(linhas_acidentes).sort_values('data').reset_index(drop=True)
acidentes.insert(0, 'acidente_id', [f'A{i:03d}' for i in range(1, len(acidentes) + 1)])

# Consolidação mensal
com_afastamento = acidentes[acidentes['tipo'] == 'Com afastamento']
mensal = horas_trabalhadas.groupby('mes', as_index=False)['hht'].sum()
mensal = mensal.merge(com_afastamento.groupby('mes').size().rename('acidentes_com_afastamento'), on='mes', how='left')
mensal = mensal.merge(acidentes[acidentes['tipo'] == 'Sem afastamento'].groupby('mes').size().rename('acidentes_sem_afastamento'), on='mes', how='left')
mensal = mensal.merge(com_afastamento.groupby('mes')[['dias_perdidos', 'dias_debitados']].sum(), on='mes', how='left')
mensal = mensal.fillna(0)
for coluna in ['acidentes_com_afastamento', 'acidentes_sem_afastamento', 'dias_perdidos', 'dias_debitados']:
    mensal[coluna] = mensal[coluna].astype(int)
mensal['tf_mes'] = (mensal['acidentes_com_afastamento'] * 1e6 / mensal['hht']).round(2)
mensal['tg_mes'] = ((mensal['dias_perdidos'] + mensal['dias_debitados']) * 1e6 / mensal['hht']).round(0)
soma_12 = mensal[['hht', 'acidentes_com_afastamento', 'dias_perdidos', 'dias_debitados']].rolling(12).sum()
mensal['tf_12m'] = (soma_12['acidentes_com_afastamento'] * 1e6 / soma_12['hht']).round(2)
mensal['tg_12m'] = ((soma_12['dias_perdidos'] + soma_12['dias_debitados']) * 1e6 / soma_12['hht']).round(0)
mensal.insert(1, 'rotulo', [rotulo_mes(m) for m in MESES])
tf_tg_mensal = mensal


def taxas(meses, setor=None):
    """Calcula TF e TG de um conjunto de meses, para a empresa ou para um setor."""
    meses = [str(m) for m in meses]
    h = horas_trabalhadas[horas_trabalhadas['mes'].isin(meses)]
    a = com_afastamento[com_afastamento['mes'].isin(meses)]
    if setor:
        h, a = h[h['setor'] == setor], a[a['setor'] == setor]
    hht = h['hht'].sum()
    return len(a) * 1e6 / hht, (a['dias_perdidos'].sum() + a['dias_debitados'].sum()) * 1e6 / hht


TF_ATUAL, TG_ATUAL = taxas(MESES_ATUAIS)
TF_ANTERIOR, TG_ANTERIOR = taxas(MESES_ANTERIORES)
TF_POR_SETOR = pd.Series({s: taxas(MESES_ATUAIS, s)[0] for s in SETORES}).sort_values()

print(f'{len(acidentes)} acidentes em {MESES_HISTORICO} meses')
print(f'TF 12 meses: {TF_ATUAL:.2f} (anterior {TF_ANTERIOR:.2f}) · TG 12 meses: {TG_ATUAL:.0f} (anterior {TG_ANTERIOR:.0f})')
tf_tg_mensal.tail()

## Etapa 5 · Montar o inventário de riscos da NR-1

Monta o inventário de riscos do PGR com dois perigos para cada combinação de setor e categoria. O nível de risco usa a **matriz 5×5**:

- **Nível** = probabilidade × severidade, de 1 a 25.
- **Grau**: Baixo (1 a 4), Moderado (5 a 9), Alto (10 a 16) e Crítico (17 a 25).

A **Conformidade NR-1** é o percentual de riscos com plano de ação concluído ou em andamento.

In [ ]:
PERIGOS = {
    'Físico': ['Ruído contínuo', 'Calor', 'Vibração', 'Iluminação deficiente', 'Radiação não ionizante'],
    'Químico': ['Vapores de solventes', 'Poeira mineral', 'Produtos de limpeza', 'Névoas de óleo', 'Fumos de solda'],
    'Biológico': ['Material biológico', 'Fungos em dutos de ar', 'Resíduos contaminados', 'Água contaminada', 'Agentes infecciosos'],
    'Ergonômico': ['Levantamento de peso', 'Postura inadequada', 'Movimentos repetitivos', 'Trabalho noturno', 'Pressão por metas'],
    'Acidentes': ['Eletricidade', 'Trabalho em altura', 'Máquinas sem proteção', 'Queda de mesmo nível', 'Circulação de empilhadeiras'],
}
# Probabilidade e severidade de referência de cada setor, por categoria
PERFIL = {
    'Administrativo': {'Físico': (2, 2), 'Químico': (1, 2), 'Biológico': (2, 2), 'Ergonômico': (3, 3), 'Acidentes': (2, 2)},
    'Produção':       {'Físico': (4, 3), 'Químico': (3, 3), 'Biológico': (1, 2), 'Ergonômico': (4, 4), 'Acidentes': (3, 4)},
    'Manutenção':     {'Físico': (3, 3), 'Químico': (3, 3), 'Biológico': (2, 2), 'Ergonômico': (3, 3), 'Acidentes': (3, 4)},
    'Logística':      {'Físico': (3, 3), 'Químico': (2, 2), 'Biológico': (2, 2), 'Ergonômico': (4, 3), 'Acidentes': (3, 4)},
    'Laboratório':    {'Físico': (2, 2), 'Químico': (4, 4), 'Biológico': (3, 3), 'Ergonômico': (2, 3), 'Acidentes': (2, 3)},
    'Atendimento':    {'Físico': (1, 2), 'Químico': (2, 2), 'Biológico': (3, 3), 'Ergonômico': (3, 3), 'Acidentes': (2, 3)},
}
STATUS_PLANO = ['Concluído', 'Em andamento', 'Atrasado', 'Sem plano']
PROB_STATUS = {
    'Crítico': [0.15, 0.45, 0.25, 0.15],
    'Alto': [0.25, 0.45, 0.20, 0.10],
    'Moderado': [0.35, 0.35, 0.15, 0.15],
    'Baixo': [0.45, 0.20, 0.10, 0.25],
}
RESPONSAVEIS = {
    'Físico': 'Higiene Ocupacional', 'Químico': 'Higiene Ocupacional', 'Biológico': 'Medicina do Trabalho',
    'Ergonômico': 'Ergonomia', 'Acidentes': 'Engenharia de Segurança',
}


def grau_risco(nivel):
    if nivel <= 4:
        return 'Baixo'
    if nivel <= 9:
        return 'Moderado'
    if nivel <= 16:
        return 'Alto'
    return 'Crítico'


linhas = []
for setor in SETORES:
    for categoria in CATEGORIAS_RISCO:
        p_base, s_base = PERFIL[setor][categoria]
        for perigo in rng.choice(PERIGOS[categoria], size=2, replace=False):
            prob = int(np.clip(p_base + rng.integers(-1, 2), 1, 5))
            sev = int(np.clip(s_base + rng.integers(-1, 2), 1, 5))
            grau = grau_risco(prob * sev)
            status = rng.choice(STATUS_PLANO, p=PROB_STATUS[grau])
            if status == 'Sem plano':
                prazo = pd.NaT
            elif status == 'Atrasado':
                prazo = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(5, 120)))
            elif status == 'Em andamento':
                prazo = DATA_REFERENCIA + pd.Timedelta(days=int(rng.integers(10, 180)))
            else:
                prazo = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(10, 300)))
            linhas.append({
                'setor': setor, 'categoria': categoria, 'perigo': perigo,
                'probabilidade': prob, 'severidade': sev, 'nivel': prob * sev, 'grau': grau,
                'status_plano': status, 'responsavel': RESPONSAVEIS[categoria], 'prazo_plano': prazo,
            })

riscos_nr1 = pd.DataFrame(linhas)
riscos_nr1.insert(0, 'risco_id', [f'R{i:03d}' for i in range(1, len(riscos_nr1) + 1)])
riscos_nr1['plano_em_dia'] = riscos_nr1['status_plano'].isin(['Concluído', 'Em andamento'])

CONFORMIDADE_NR1 = 100 * riscos_nr1['plano_em_dia'].mean()
riscos_sem_plano = (riscos_nr1[riscos_nr1['grau'].isin(['Alto', 'Crítico']) & ~riscos_nr1['plano_em_dia']]
                    .sort_values(['nivel', 'risco_id'], ascending=[False, True]))

print(f'{len(riscos_nr1)} riscos · conformidade NR-1: {CONFORMIDADE_NR1:.1f}%')
print(riscos_nr1['grau'].value_counts().to_string())

## Etapa 6 · Gerar a brigada de incêndio e verificar a cobertura

Seleciona os brigadistas entre os colaboradores de cada prédio e turno e registra a última reciclagem, que é **anual**. A cobertura exige pelo menos **3 brigadistas com reciclagem válida em cada prédio e turno**.

In [ ]:
linhas = []
for predio in PREDIOS:
    for turno in TURNOS:
        grupo = colaboradores[(colaboradores['predio'] == predio) & (colaboradores['turno'] == turno)]
        if grupo.empty:
            grupo = colaboradores[colaboradores['turno'] == turno]
        quantidade = min(len(grupo), int(rng.integers(3, 6)))
        for _, pessoa in grupo.sample(quantidade, random_state=int(rng.integers(1e9))).iterrows():
            treinamento = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(0, 400)))
            linhas.append({
                'colaborador_id': pessoa['colaborador_id'], 'setor': pessoa['setor'],
                'predio': predio, 'turno': turno, 'ultimo_treinamento': treinamento,
                'exame_especifico': rng.choice(['Apto', 'Pendente'], p=[0.9, 0.1]),
            })

brigada = pd.DataFrame(linhas)
brigada.insert(0, 'brigadista_id', [f'B{i:03d}' for i in range(1, len(brigada) + 1)])
brigada['vencimento_reciclagem'] = brigada['ultimo_treinamento'] + pd.Timedelta(days=365)
brigada['dias_para_vencer'] = (brigada['vencimento_reciclagem'] - DATA_REFERENCIA).dt.days
brigada['situacao'] = brigada['dias_para_vencer'].apply(faixa_prazo)
brigada['valido'] = brigada['situacao'] != 'Vencido'

cobertura = (brigada.groupby(['predio', 'turno'])
             .agg(brigadistas=('brigadista_id', 'count'), validos=('valido', 'sum'))
             .reindex(pd.MultiIndex.from_product([PREDIOS, TURNOS], names=['predio', 'turno']), fill_value=0)
             .reset_index())
cobertura['abaixo_do_minimo'] = cobertura['validos'] < BRIGADISTAS_MINIMOS

BRIGADA_VALIDA = 100 * brigada['valido'].mean()
print(f'{len(brigada)} brigadistas · {BRIGADA_VALIDA:.1f}% com reciclagem válida · '
      f'{cobertura["abaixo_do_minimo"].sum()} prédio(s)/turno(s) abaixo do mínimo')
cobertura

## Etapa 7 · Gerar o inventário de equipamentos de segurança

Registra extintores, hidrantes, iluminação de emergência, kits de primeiros socorros e EPIs críticos, com a data de validade (recarga, inspeção ou vencimento do lote). Cada item recebe uma faixa de prazo: vencido, vence em até 30 dias, vence em 31 a 60 dias ou em dia.

In [ ]:
TIPOS_EQUIPAMENTO = {
    'Extintor PQS': 30,
    'Extintor CO2': 15,
    'Extintor água': 10,
    'Hidrante (mangueira)': 9,
    'Luz de emergência': 18,
    'Kit de primeiros socorros': 9,
    'Cinto paraquedista (EPI)': 12,
    'Respirador PFF2 – lote (EPI)': 9,
    'Chuveiro lava-olhos': 6,
}
linhas = []
for tipo, quantidade in TIPOS_EQUIPAMENTO.items():
    for _ in range(quantidade):
        linhas.append({
            'tipo': tipo,
            'predio': rng.choice(PREDIOS),
            'localizacao': rng.choice(['Térreo', '1º andar', '2º andar']),
            'validade': DATA_REFERENCIA + pd.Timedelta(days=int(rng.integers(-40, 365))),
        })

equipamentos = pd.DataFrame(linhas)
equipamentos.insert(0, 'equipamento_id', [f'EQ{i:03d}' for i in range(1, len(equipamentos) + 1)])
equipamentos['dias_para_vencer'] = (equipamentos['validade'] - DATA_REFERENCIA).dt.days
equipamentos['situacao'] = equipamentos['dias_para_vencer'].apply(faixa_prazo)

EQUIPAMENTOS_VALIDOS = 100 * (equipamentos['situacao'] != 'Vencido').mean()
print(f'{len(equipamentos)} equipamentos · {EQUIPAMENTOS_VALIDOS:.1f}% dentro da validade')
print(equipamentos['situacao'].value_counts().reindex(FAIXAS, fill_value=0).to_string())

## Etapa 8 · Registrar os documentos legais e o fluxo dos POPs e do PAE

Registra a vigência dos documentos legais obrigatórios (PGR, PCMSO, LTCAT, AET, prontuário das instalações elétricas e projeto de incêndio) e a etapa de revisão de cada POP e plano de atendimento a emergências (PAE).

Um POP ou PAE é considerado **parado** quando está há mais de 60 dias na mesma etapa, exceto os vigentes.

In [ ]:
# Documento, norma, responsável, validade em anos e dias até o vencimento na data de referência
DOCUMENTOS = [
    ('LTCAT – Laudo Técnico das Condições Ambientais', 'Previdência Social', 'Engenharia de Segurança', 2, -20),
    ('PCMSO – Relatório analítico', 'NR-7', 'Medicina do Trabalho', 1, 25),
    ('AET – Análise Ergonômica (Administrativo)', 'NR-17', 'Ergonomia', 2, 45),
    ('Prontuário das Instalações Elétricas', 'NR-10', 'Manutenção', 2, 330),
    ('PGR – Programa de Gerenciamento de Riscos', 'NR-1', 'Engenharia de Segurança', 2, 430),
    ('AET – Análise Ergonômica (Produção)', 'NR-17', 'Ergonomia', 2, 550),
    ('Projeto de Prevenção e Combate a Incêndio', 'NR-23 / Corpo de Bombeiros', 'Engenharia de Segurança', 3, 610),
]
linhas = []
for documento, norma, responsavel, anos, dias in DOCUMENTOS:
    vencimento = DATA_REFERENCIA + pd.Timedelta(days=dias + int(rng.integers(-3, 4)))
    linhas.append({
        'documento': documento, 'norma': norma, 'responsavel': responsavel,
        'emissao': vencimento - pd.DateOffset(years=anos), 'vencimento': vencimento,
    })
documentos_legais = pd.DataFrame(linhas)
documentos_legais['dias_para_vencer'] = (documentos_legais['vencimento'] - DATA_REFERENCIA).dt.days
documentos_legais['situacao'] = documentos_legais['dias_para_vencer'].apply(faixa_prazo)
documentos_legais = documentos_legais.sort_values('dias_para_vencer').reset_index(drop=True)
DOCUMENTOS_VIGENTES = 100 * (documentos_legais['situacao'] != 'Vencido').mean()

FLUXOS = [
    ('POP-SST-001', 'Trabalho em altura', 'POP'),
    ('POP-SST-002', 'Bloqueio e etiquetagem (LOTO)', 'POP'),
    ('POP-SST-003', 'Espaço confinado', 'POP'),
    ('POP-SST-004', 'Seleção e uso de EPI', 'POP'),
    ('POP-SST-005', 'Investigação de acidentes', 'POP'),
    ('POP-SST-006', 'Permissão de trabalho', 'POP'),
    ('POP-SST-007', 'Movimentação de cargas', 'POP'),
    ('POP-SST-008', 'Produtos químicos e FDS', 'POP'),
    ('POP-SST-009', 'Comunicação de acidente (CAT)', 'POP'),
    ('POP-SST-010', 'Inspeção de extintores', 'POP'),
    ('PAE-01', 'Incêndio e abandono de área', 'PAE'),
    ('PAE-02', 'Vazamento de produto químico', 'PAE'),
    ('PAE-03', 'Primeiros socorros e remoção', 'PAE'),
    ('PAE-04', 'Falha elétrica e apagão', 'PAE'),
]
ETAPAS_FLUXO = ['Em elaboração', 'Em revisão', 'Validação – Engenharia', 'Validação – Diretoria', 'Vigente', 'Vencido']
RESPONSAVEL_ETAPA = {
    'Em elaboração': 'Técnico de Segurança', 'Em revisão': 'Coordenação de SST',
    'Validação – Engenharia': 'Engenharia de Segurança', 'Validação – Diretoria': 'Diretoria',
    'Vigente': '—', 'Vencido': 'Coordenação de SST',
}
linhas = []
for codigo, titulo, tipo in FLUXOS:
    etapa = rng.choice(ETAPAS_FLUXO, p=[0.05, 0.10, 0.12, 0.08, 0.50, 0.15])
    if etapa == 'Vigente':
        ultima_revisao = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(30, 650)))
        dias_etapa = (DATA_REFERENCIA - ultima_revisao).days
    else:
        dias_etapa = int(rng.integers(5, 140))
        ultima_revisao = DATA_REFERENCIA - pd.Timedelta(days=int(rng.integers(400, 900)))
    if etapa == 'Vencido':
        ultima_revisao = DATA_REFERENCIA - pd.Timedelta(days=730 + dias_etapa)
    linhas.append({
        'codigo': codigo, 'titulo': titulo, 'tipo': tipo, 'etapa': etapa,
        'responsavel_etapa': RESPONSAVEL_ETAPA[etapa],
        'data_entrada_etapa': DATA_REFERENCIA - pd.Timedelta(days=dias_etapa), 'dias_na_etapa': dias_etapa,
        'ultima_revisao': ultima_revisao, 'proxima_revisao': ultima_revisao + pd.DateOffset(years=2),
    })
pops_pae = pd.DataFrame(linhas)
pops_pae['parado'] = (pops_pae['etapa'] != 'Vigente') & (pops_pae['dias_na_etapa'] > DIAS_FLUXO_PARADO)
POPS_VIGENTES = 100 * (pops_pae['etapa'] == 'Vigente').mean()

print(f'Documentos vigentes: {DOCUMENTOS_VIGENTES:.1f}% · POPs e PAE vigentes: {POPS_VIGENTES:.1f}%')
print(pops_pae['etapa'].value_counts().reindex(ETAPAS_FLUXO, fill_value=0).to_string())
documentos_legais

## Etapa 9 · Simular os ciclos da pesquisa de percepção

Simula quatro ciclos trimestrais da pesquisa de clima de segurança. Cada participante responde às seis dimensões em escala de **1 (discordo totalmente) a 5 (concordo totalmente)**. As notas 4 e 5 contam como **favoráveis**.

In [ ]:
PARTICIPACAO_CICLO = [0.58, 0.63, 0.66, 0.71]
# Média da escala latente no primeiro e no último ciclo, por dimensão
MEDIA_DIMENSAO = {
    'Eficácia dos treinamentos': (4.05, 4.22),
    'Liderança – saúde física': (3.55, 4.12),
    'Comunicação sobre riscos': (3.90, 4.05),
    'Confiança para relatar incidentes': (3.70, 3.78),
    'Direito de recusa': (3.65, 3.74),
    'Liderança – saúde mental': (3.05, 3.12),
}
EFEITO_SETOR = {'Administrativo': 0.12, 'Produção': -0.15, 'Manutenção': -0.10,
                'Logística': -0.05, 'Laboratório': 0.08, 'Atendimento': 0.05}
ESCALA = {1: 'Discordo totalmente', 2: 'Discordo', 3: 'Neutro', 4: 'Concordo', 5: 'Concordo totalmente'}

linhas = []
for i, ciclo in enumerate(CICLOS):
    respondentes = colaboradores.sample(round(EFETIVO * PARTICIPACAO_CICLO[i]), random_state=SEMENTE + i)
    for n, (_, pessoa) in enumerate(respondentes.iterrows(), start=1):
        for dimensao, (inicio, fim) in MEDIA_DIMENSAO.items():
            media = inicio + (fim - inicio) * i / (len(CICLOS) - 1) + EFEITO_SETOR[pessoa['setor']]
            nota = int(np.clip(np.rint(rng.normal(media, 1.0)), 1, 5))
            linhas.append({'ciclo': ciclo, 'resposta_id': f'{i + 1}-{n:04d}', 'setor': pessoa['setor'],
                           'dimensao': dimensao, 'nota': nota})
pesquisa_respostas = pd.DataFrame(linhas)


def resumir(respostas):
    contagem = respostas['nota'].value_counts(normalize=True).reindex(range(1, 6), fill_value=0) * 100
    resumo = {ESCALA[k]: round(v, 1) for k, v in contagem.items()}
    resumo['favoravel'] = round(contagem[4] + contagem[5], 1)
    resumo['respostas'] = len(respostas)
    return resumo


linhas = []
for i, ciclo in enumerate(CICLOS):
    do_ciclo = pesquisa_respostas[pesquisa_respostas['ciclo'] == ciclo]
    participantes = do_ciclo['resposta_id'].nunique()
    for dimensao in DIMENSOES + ['Geral (satisfação em SST)']:
        recorte = do_ciclo if dimensao.startswith('Geral') else do_ciclo[do_ciclo['dimensao'] == dimensao]
        linhas.append({'ciclo': ciclo, 'dimensao': dimensao, 'participantes': participantes,
                       'participacao': round(100 * participantes / EFETIVO, 1), **resumir(recorte)})
pesquisa_resumo = pd.DataFrame(linhas)

ultimo = pesquisa_resumo[pesquisa_resumo['ciclo'] == CICLOS[-1]].set_index('dimensao')
penultimo = pesquisa_resumo[pesquisa_resumo['ciclo'] == CICLOS[-2]].set_index('dimensao')
primeiro = pesquisa_resumo[pesquisa_resumo['ciclo'] == CICLOS[0]].set_index('dimensao')
SATISFACAO = ultimo.loc['Geral (satisfação em SST)', 'favoravel']
SATISFACAO_ANTERIOR = penultimo.loc['Geral (satisfação em SST)', 'favoravel']
PARTICIPACAO = ultimo.loc['Geral (satisfação em SST)', 'participacao']
PARTICIPANTES = int(ultimo.loc['Geral (satisfação em SST)', 'participantes'])
FAVORAVEL = ultimo.loc[DIMENSOES, 'favoravel']

print(f'Ciclo {CICLOS[-1]}: participação {PARTICIPACAO:.0f}% · satisfação {SATISFACAO:.1f}%')
FAVORAVEL.sort_values()

## Etapa 10 · Cruzar os indicadores com os selos e certificações

Reúne os indicadores calculados e compara cada um com a meta, para três selos: **Great Place to Work (GPTW)**, **ISO 45001** e **ESG**.

- **OK**: meta atingida.
- **Atenção**: até 10% distante da meta.
- **Alerta Vermelho**: mais de 10% distante da meta.

Um selo fica **Em risco** se tiver ao menos um indicador em alerta vermelho, **Atenção** se tiver indicador em atenção e **Pronto** se todos estiverem na meta.

In [ ]:
INDICADORES = [
    # indicador, selo, critério, valor atual, meta, sentido, unidade, plano de mitigação
    ('Confiança para relatar incidentes', 'GPTW', 'Credibilidade', FAVORAVEL['Confiança para relatar incidentes'], METAS['favoravel'], '>=', '%',
     'Dar retorno formal aos relatos recebidos e garantir canal sem retaliação.'),
    ('Direito de recusa', 'GPTW', 'Respeito', FAVORAVEL['Direito de recusa'], METAS['favoravel'], '>=', '%',
     'Comunicar o direito de recusa e capacitar lideranças para acolher recusas.'),
    ('Liderança – saúde mental', 'GPTW', 'Respeito', FAVORAVEL['Liderança – saúde mental'], METAS['favoravel'], '>=', '%',
     'Capacitar lideranças em segurança psicológica e divulgar canais de apoio.'),
    ('Satisfação geral em SST', 'GPTW', 'Orgulho', SATISFACAO, METAS['favoravel'], '>=', '%',
     'Divulgar resultados e conquistas de SST e reconhecer boas práticas das equipes.'),
    ('Taxa de Frequência', 'ISO 45001', '9.1 Monitoramento e medição', TF_ATUAL, METAS['tf'], '<=', '',
     'Investigar as causas dos acidentes com afastamento e reforçar inspeções nos setores críticos.'),
    ('Taxa de Gravidade', 'ISO 45001', '9.1 Monitoramento e medição', TG_ATUAL, METAS['tg'], '<=', '',
     'Priorizar controles para os perigos de maior severidade e acompanhar o retorno ao trabalho.'),
    ('Riscos com plano de ação (NR-1)', 'ISO 45001', '6.1 Ações para abordar riscos', CONFORMIDADE_NR1, METAS['conformidade_nr1'], '>=', '%',
     'Definir responsáveis e prazos para os riscos sem plano, começando pelos críticos.'),
    ('Documentos legais vigentes', 'ISO 45001', '7.5 Informação documentada', DOCUMENTOS_VIGENTES, METAS['documentos_vigentes'], '>=', '%',
     'Instaurar a revisão dos documentos vencidos e manter agenda de vigência.'),
    ('Brigadistas com reciclagem válida', 'ISO 45001', '8.2 Preparação e resposta a emergências', BRIGADA_VALIDA, METAS['brigada_valida'], '>=', '%',
     'Programar a reciclagem dos brigadistas vencidos e recompor os turnos abaixo do mínimo.'),
    ('Equipamentos dentro da validade', 'ISO 45001', '8.2 Preparação e resposta a emergências', EQUIPAMENTOS_VALIDOS, METAS['equipamentos_validos'], '>=', '%',
     'Substituir ou recarregar os itens vencidos e antecipar os que vencem em 30 dias.'),
    ('ASOs periódicos em dia', 'ISO 45001', '8.1 Controle operacional', aso_em_dia, METAS['aso_em_dia'], '>=', '%',
     'Convocar os colaboradores com ASO vencido e escalonar a agenda dos próximos vencimentos.'),
    ('Eficácia dos treinamentos', 'ISO 45001', '7.2 Competência', FAVORAVEL['Eficácia dos treinamentos'], METAS['favoravel'], '>=', '%',
     'Avaliar a reação e a aplicação prática dos treinamentos e revisar os conteúdos com pior avaliação.'),
    ('POPs e PAE vigentes', 'ESG', 'Governança – procedimentos', POPS_VIGENTES, METAS['pops_vigentes'], '>=', '%',
     'Definir prazo para validação dos fluxos pendentes na Engenharia e na Diretoria.'),
    ('Participação na pesquisa', 'ESG', 'Social – escuta dos colaboradores', PARTICIPACAO, METAS['participacao'], '>=', '%',
     'Ampliar a divulgação da pesquisa e garantir tempo para resposta durante a jornada.'),
]


def avaliar(atual, meta, sentido):
    """Retorna a distância relativa até a meta, a situação e o atingimento (100% = meta cumprida)."""
    if sentido == '>=':
        distancia = (meta - atual) / meta
        atingimento = 100 * atual / meta
    else:
        distancia = (atual - meta) / meta
        atingimento = 100 * meta / atual if atual else 100.0
    if distancia <= 0:
        situacao = 'OK'
    elif distancia <= 0.10:
        situacao = 'Atenção'
    else:
        situacao = 'Alerta Vermelho'
    return distancia, situacao, atingimento


selos_certificacoes = pd.DataFrame(INDICADORES, columns=['indicador', 'selo', 'criterio', 'atual', 'meta', 'sentido', 'unidade', 'plano_mitigacao'])
avaliacao = selos_certificacoes.apply(lambda l: avaliar(l['atual'], l['meta'], l['sentido']), axis=1, result_type='expand')
selos_certificacoes[['distancia_meta', 'situacao', 'atingimento']] = avaliacao
selos_certificacoes['atual'] = selos_certificacoes['atual'].astype(float).round(2)
selos_certificacoes['distancia_meta'] = (selos_certificacoes['distancia_meta'].astype(float) * 100).round(1)
selos_certificacoes['atingimento'] = selos_certificacoes['atingimento'].astype(float).round(1)

SELOS = ['GPTW', 'ISO 45001', 'ESG']
resumo_selos = []
for selo in SELOS:
    s = selos_certificacoes[selos_certificacoes['selo'] == selo]['situacao']
    alerta, atencao = (s == 'Alerta Vermelho').sum(), (s == 'Atenção').sum()
    resumo_selos.append({'selo': selo, 'indicadores': len(s), 'ok': int((s == 'OK').sum()),
                         'atencao': int(atencao), 'alerta': int(alerta),
                         'situacao': 'Em risco' if alerta else ('Atenção' if atencao else 'Pronto')})
resumo_selos = pd.DataFrame(resumo_selos)


def situacao_de(indicador):
    return selos_certificacoes.set_index('indicador').loc[indicador, 'situacao']


print(resumo_selos.to_string(index=False))
selos_certificacoes[['indicador', 'selo', 'atual', 'meta', 'situacao']]

## Etapa 11 · Aplicar a identidade visual

Define as cores, a fonte e um modelo (*template*) do Plotly aplicado a todos os gráficos. As cores seguem o padrão do painel: azul-marinho no cabeçalho, azul para destaque e vermelho, laranja, amarelo e verde para as situações.

In [ ]:
CORES = {
    'marinho': '#0F3466', 'azul': '#2B78D6', 'azul_claro': '#A3C6F2', 'azul_fundo': '#E8F1FB',
    'texto': '#252423', 'texto_2': '#605E5C', 'cinza': '#D6D5CF', 'grade': '#ECEBE6', 'eixo': '#C8C6C0',
    'vermelho': '#D13438', 'vermelho_claro': '#F2A7A7', 'laranja': '#EE8660', 'amarelo': '#F7B21B', 'verde': '#2E8B3E',
}
COR_FAIXA = {'Vencido': CORES['vermelho'], 'Vence em até 30 dias': CORES['laranja'],
             'Vence em 31 a 60 dias': CORES['amarelo'], 'Em dia': CORES['azul_claro']}
COR_GRAU = {'Baixo': '#D5EDD6', 'Moderado': '#FCEBB6', 'Alto': '#F6C9B5', 'Crítico': '#F0A8A8'}
COR_SITUACAO = {'OK': CORES['verde'], 'Atenção': CORES['amarelo'], 'Alerta Vermelho': CORES['vermelho']}
COR_LIKERT = {'Discordo totalmente': CORES['vermelho'], 'Discordo': CORES['vermelho_claro'], 'Neutro': CORES['cinza'],
              'Concordo': CORES['azul_claro'], 'Concordo totalmente': CORES['azul']}
CORES_SERIES = ['#2B78D6', '#E8703A', '#2E8B3E', '#8E5CC4', '#C2185B', '#00897B']
FONTE = 'Inter, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif'

pio.templates['sst'] = go.layout.Template(layout=dict(
    font=dict(family=FONTE, size=12, color=CORES['texto']),
    paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
    separators=',.', colorway=CORES_SERIES,
    margin=dict(l=10, r=20, t=40, b=10),
    hoverlabel=dict(bgcolor='white', bordercolor=CORES['eixo'], font=dict(family=FONTE, size=12, color=CORES['texto'])),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0, font=dict(size=12), bgcolor='rgba(0,0,0,0)'),
    modebar=dict(bgcolor='rgba(0,0,0,0)', color='#B3B0AD', activecolor='#2B78D6', orientation='v'),
    xaxis=dict(showgrid=False, linecolor=CORES['eixo'], tickcolor=CORES['eixo'], zeroline=False, automargin=True,
               tickfont=dict(color=CORES['texto_2'])),
    yaxis=dict(gridcolor=CORES['grade'], zeroline=False, automargin=True, tickfont=dict(color=CORES['texto_2'])),
))
pio.templates.default = 'plotly_white+sst'

CONFIG_GRAFICO = {
    'displaylogo': False, 'responsive': True,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d', 'toggleSpikelines'],
    'toImageButtonOptions': {'format': 'png', 'scale': 2},
}


def num(valor, casas=0):
    """Formata número no padrão brasileiro: 1.234,56."""
    texto = f'{valor:,.{casas}f}'
    return texto.replace(',', '§').replace('.', ',').replace('§', '.')


def pct(valor, casas=0):
    return f'{num(valor, casas)}%'


def data_br(valor):
    return '—' if pd.isna(valor) else pd.Timestamp(valor).strftime('%d/%m/%Y')


def minuscula(texto):
    """Coloca só a primeira letra em minúscula, preservando siglas como SST e NR-1."""
    return texto if texto[:2].isupper() else texto[:1].lower() + texto[1:]


def texto_prazo(dias):
    if dias < 0:
        return f'vencido há {-dias} dias'
    if dias <= 60:
        return f'vence em {dias} dias'
    return f'vence em {round(dias / 30.44)} meses'


print('Identidade visual aplicada.')

## Etapa 12 · Preparar os cards, os alertas e a leitura rápida

Monta os componentes em HTML: os cards de indicador (com meta, comparação e minigráfico), a lista de alertas ordenada por gravidade e a **Leitura rápida** de cada aba, uma frase escrita automaticamente a partir dos dados.

In [ ]:
ICONE = {'ok': '✓', 'atencao': '!', 'laranja': '!', 'alerta': '✕'}
CLASSE_SITUACAO = {'OK': 'ok', 'Atenção': 'atencao', 'Alerta Vermelho': 'alerta', 'Em risco': 'alerta', 'Pronto': 'ok',
                   'Vencido': 'alerta', 'Vence em até 30 dias': 'laranja', 'Vence em 31 a 60 dias': 'atencao', 'Em dia': 'ok',
                   'Sem plano': 'alerta', 'Atrasado': 'laranja', 'Em andamento': 'atencao', 'Concluído': 'ok',
                   'Apto': 'ok', 'Pendente': 'atencao'}


def selo_html(texto, classe=None):
    classe = classe or CLASSE_SITUACAO.get(texto, 'ok')
    return f'<span class="pill {classe}"><span class="ic">{ICONE[classe]}</span>{html.escape(texto)}</span>'


def minigrafico(valores, largura=130, altura=40):
    v = np.asarray(valores, dtype=float)
    menor, maior = v.min(), v.max()
    if maior == menor:
        maior = menor + 1
    xs = np.linspace(4, largura - 6, len(v))
    ys = altura - 5 - (v - menor) / (maior - menor) * (altura - 10)
    pontos = ' '.join(f'{x:.1f},{y:.1f}' for x, y in zip(xs, ys))
    return (f'<svg class="mini" width="{largura}" height="{altura}" viewBox="0 0 {largura} {altura}" aria-hidden="true">'
            f'<polyline points="{pontos}" fill="none" stroke="{CORES["cinza"]}" stroke-width="2"/>'
            f'<circle cx="{xs[-1]:.1f}" cy="{ys[-1]:.1f}" r="3.5" fill="{CORES["azul"]}"/></svg>')


def variacao_html(delta, menor_e_melhor, sufixo, casas):
    if abs(delta) < 10 ** -casas / 2:
        return '<span class="var neutra">= estável</span>'
    melhorou = (delta < 0) if menor_e_melhor else (delta > 0)
    seta = '▼' if delta < 0 else '▲'
    return f'<span class="var {"boa" if melhorou else "ruim"}">{seta} {num(abs(delta), casas)}{sufixo}</span>'


def card_kpi(titulo, valor, unidade, situacao, meta_texto, rodape, grafico=''):
    return f'''<div class="kpi"><div class="kpi-topo"><span class="kpi-titulo">{titulo}</span>{selo_html(situacao)}</div>
      <div class="kpi-meio"><div><div class="kpi-valor">{valor}</div><div class="kpi-unidade">{unidade}</div></div>{grafico}</div>
      <div class="kpi-rodape"><span>{meta_texto}</span>{rodape}</div></div>'''


def leitura(texto):
    return f'<div class="leitura"><span class="leitura-rotulo">Leitura rápida</span><p>{texto}</p></div>'


def cartao(titulo, subtitulo, conteudo, classe=''):
    sub = f'<p class="sub">{subtitulo}</p>' if subtitulo else ''
    return f'<section class="cartao {classe}"><h3>{titulo}</h3>{sub}{conteudo}</section>'


def tabela(df, colunas, numericas=(), quebra=()):
    def classe(c):
        return ' '.join(k for k, sim in [('n', c in numericas), ('q', c in quebra)] if sim)
    cab = ''.join(f'<th class="{classe(c)}">{html.escape(r)}</th>' for c, r in colunas.items())
    linhas_html = []
    for _, linha in df.iterrows():
        celulas = ''.join(f'<td class="{classe(c)}">{linha[c]}</td>' for c in colunas)
        linhas_html.append(f'<tr>{celulas}</tr>')
    if not linhas_html:
        linhas_html = [f'<tr><td colspan="{len(colunas)}" class="vazio">Nenhum item nesta situação.</td></tr>']
    return f'<div class="tabela"><table><thead><tr>{cab}</tr></thead><tbody>{"".join(linhas_html)}</tbody></table></div>'


# ---------- Cards da Visão Geral ----------
tf_12 = tf_tg_mensal['tf_12m'].iloc[-12:]
tg_12 = tf_tg_mensal['tg_12m'].iloc[-12:]
favoravel_ciclos = pesquisa_resumo[pesquisa_resumo['dimensao'].str.startswith('Geral')]['favoravel'].tolist()
sem_plano_altos = len(riscos_sem_plano)

progresso_nr1 = f'''<div class="barra"><div class="barra-cheia" style="width:{min(CONFORMIDADE_NR1, 100):.1f}%"></div>
  <div class="barra-meta" style="left:{METAS['conformidade_nr1']}%"></div></div>'''
cards_visao = ''.join([
    card_kpi('Taxa de Frequência', num(TF_ATUAL, 2), 'por milhão de HHT', situacao_de('Taxa de Frequência'),
             f'Meta ≤ {num(METAS["tf"])}', variacao_html(TF_ATUAL - TF_ANTERIOR, True, ' vs 12 meses anteriores', 2), minigrafico(tf_12)),
    card_kpi('Taxa de Gravidade', num(TG_ATUAL), 'dias por milhão de HHT', situacao_de('Taxa de Gravidade'),
             f'Meta ≤ {num(METAS["tg"])}', variacao_html(TG_ATUAL - TG_ANTERIOR, True, ' vs 12 meses anteriores', 0), minigrafico(tg_12)),
    card_kpi('Conformidade NR-1', pct(CONFORMIDADE_NR1), 'riscos com plano de ação', situacao_de('Riscos com plano de ação (NR-1)'),
             f'Meta ≥ {METAS["conformidade_nr1"]}%', f'<span class="var neutra">{sem_plano_altos} riscos altos ou críticos sem plano em dia</span>',
             progresso_nr1),
    card_kpi('Satisfação em SST', pct(SATISFACAO), f'favoráveis no {CICLOS[-1]}', situacao_de('Satisfação geral em SST'),
             f'Meta ≥ {METAS["favoravel"]}%', variacao_html(SATISFACAO - SATISFACAO_ANTERIOR, False, ' p.p. vs ciclo anterior', 1),
             minigrafico(favoravel_ciclos)),
])

# ---------- Alertas ----------
contagem = {
    'criticos_sem_plano': int(((riscos_nr1['grau'] == 'Crítico') & ~riscos_nr1['plano_em_dia']).sum()),
    'turnos_abaixo': int(cobertura['abaixo_do_minimo'].sum()),
    'equip_vencidos': int((equipamentos['situacao'] == 'Vencido').sum()),
    'equip_30': int((equipamentos['situacao'] == 'Vence em até 30 dias').sum()),
    'brig_vencidos': int((brigada['situacao'] == 'Vencido').sum()),
    'brig_30': int((brigada['situacao'] == 'Vence em até 30 dias').sum()),
    'docs_vencidos': int((documentos_legais['situacao'] == 'Vencido').sum()),
    'docs_60': int(documentos_legais['situacao'].isin(['Vence em até 30 dias', 'Vence em 31 a 60 dias']).sum()),
    'pops_vencidos': int((pops_pae['etapa'] == 'Vencido').sum()),
    'pops_validacao': int(pops_pae['etapa'].isin(['Validação – Engenharia', 'Validação – Diretoria']).sum()),
    'pops_parados': int(pops_pae['parado'].sum()),
    'ind_alerta': int((selos_certificacoes['situacao'] == 'Alerta Vermelho').sum()),
}
REGRAS_ALERTA = [
    ('alerta', 'criticos_sem_plano', 2, '{n} risco(s) <b>crítico(s)</b> sem plano de ação em dia'),
    ('alerta', 'turnos_abaixo', 2, f'{{n}} prédio(s)/turno(s) com menos de {BRIGADISTAS_MINIMOS} brigadistas válidos'),
    ('alerta', 'equip_vencidos', 2, '{n} equipamento(s) de segurança <b>vencido(s)</b>'),
    ('alerta', 'brig_vencidos', 2, '{n} brigadista(s) com reciclagem <b>vencida</b>'),
    ('alerta', 'docs_vencidos', 3, '{n} documento(s) legal(is) <b>vencido(s)</b>'),
    ('alerta', 'pops_vencidos', 3, '{n} POP/PAE <b>vencido(s)</b>'),
    ('alerta', 'ind_alerta', 5, '{n} indicador(es) em <b>Alerta Vermelho</b> para selos e certificações'),
    ('atencao', 'equip_30', 2, '{n} equipamento(s) vencem nos próximos 30 dias'),
    ('atencao', 'brig_30', 2, '{n} brigadista(s) com reciclagem vencendo em até 30 dias'),
    ('atencao', 'docs_60', 3, '{n} documento(s) legal(is) vencem nos próximos 60 dias'),
    ('atencao', 'pops_validacao', 3, '{n} POP/PAE aguardando validação da Engenharia ou da Diretoria'),
    ('atencao', 'pops_parados', 3, f'{{n}} POP/PAE parado(s) há mais de {DIAS_FLUXO_PARADO} dias na mesma etapa'),
]
alertas = [{'classe': c, 'aba': aba, 'texto': t.format(n=contagem[chave])}
           for c, chave, aba, t in REGRAS_ALERTA if contagem[chave] > 0]
ALERTAS_POR_ABA = {aba: sum(1 for a in alertas if a['aba'] == aba and a['classe'] == 'alerta') for aba in range(1, 6)}
itens = ''.join(f'<li><span class="pill-ic {a["classe"]}">{ICONE[a["classe"]]}</span><span class="txt">{a["texto"]}</span>'
                f'<a href="#aba-{a["aba"]}" onclick="abrirAba({a["aba"]}); return false;">Ver aba {a["aba"]} →</a></li>' for a in alertas)
lista_alertas = f'<ul class="alertas">{itens}</ul>' if alertas else '<p class="sub">Nenhum alerta no momento.</p>'

# ---------- Leitura rápida de cada aba ----------
setores_acima = TF_POR_SETOR[TF_POR_SETOR > METAS['tf']].sort_values(ascending=False).index.tolist()
frase_setores = (f'<b>{" e ".join(setores_acima)}</b> concentra{"m" if len(setores_acima) > 1 else ""} a frequência acima da meta.'
                 if setores_acima else 'Nenhum setor está acima da meta de frequência.')
LEITURA = {}
LEITURA[1] = (f'A TF acumulada em 12 meses é <b>{num(TF_ATUAL, 2)}</b> '
              f'({"dentro da" if TF_ATUAL <= METAS["tf"] else "acima da"} meta de {num(METAS["tf"])}) e '
              f'{"caiu" if TF_ATUAL < TF_ANTERIOR else "subiu"} em relação aos 12 meses anteriores. {frase_setores}')

maior = riscos_nr1.sort_values(['nivel', 'risco_id'], ascending=[False, True]).iloc[0]
n_criticos = int((riscos_nr1['grau'] == 'Crítico').sum())
LEITURA[2] = (f'{n_criticos} risco(s) crítico(s) mapeado(s); o maior nível é <b>{minuscula(maior["perigo"])}</b> em {maior["setor"]} '
              f'(nível {maior["nivel"]}). '
              + (f'{contagem["turnos_abaixo"]} prédio(s)/turno(s) estão abaixo do mínimo de brigadistas'
                 if contagem['turnos_abaixo'] else 'Todos os prédios e turnos têm o mínimo de brigadistas')
              + f' e {contagem["equip_vencidos"]} equipamento(s) estão vencidos.')

n_vigentes = int((pops_pae['etapa'] == 'Vigente').sum())
LEITURA[3] = (f'{contagem["docs_vencidos"]} documento(s) legal(is) vencido(s) e {contagem["docs_60"]} a vencer em até 60 dias. '
              f'Dos {len(pops_pae)} POPs e PAE, <b>{n_vigentes} estão vigentes</b> e {contagem["pops_parados"]} estão parados há mais de '
              f'{DIAS_FLUXO_PARADO} dias.')

evolucao = (ultimo.loc[DIMENSOES, 'favoravel'] - primeiro.loc[DIMENSOES, 'favoravel'])
participacao_inicial = primeiro.loc['Geral (satisfação em SST)', 'participacao']
LEITURA[4] = (f'<b>{FAVORAVEL.idxmin()}</b> é a dimensão mais frágil ({pct(FAVORAVEL.min())} favorável), e '
              f'<b>{evolucao.idxmax()}</b> foi a que mais evoluiu ({"+" if evolucao.max() >= 0 else ""}{num(evolucao.max())} p.p. em um ano). '
              f'A participação {"subiu" if PARTICIPACAO >= participacao_inicial else "caiu"} para {pct(PARTICIPACAO)} dos colaboradores.')

em_alerta = selos_certificacoes[selos_certificacoes['situacao'] == 'Alerta Vermelho']
if len(em_alerta):
    selo_exposto = em_alerta['selo'].value_counts().idxmax()
    puxadores = em_alerta[em_alerta['selo'] == selo_exposto].sort_values('distancia_meta', ascending=False)['indicador'].map(minuscula).tolist()[:3]
    LEITURA[5] = (f'{len(em_alerta)} de {len(selos_certificacoes)} indicadores estão em Alerta Vermelho. <b>{selo_exposto}</b> é o selo mais '
                  f'exposto, puxado principalmente por {", ".join(puxadores[:-1]) + " e " + puxadores[-1] if len(puxadores) > 1 else puxadores[0]}.')
else:
    LEITURA[5] = f'Nenhum dos {len(selos_certificacoes)} indicadores está em Alerta Vermelho.'

for aba, texto in LEITURA.items():
    print(f'Aba {aba}: {html.unescape(texto).replace("<b>", "").replace("</b>", "")}')

## Etapa 13 · Criar os gráficos

Cria todos os gráficos com Plotly. Cada gráfico tem dicas ao passar o mouse e zoom, e todos usam a identidade visual da Etapa 11.

In [ ]:
GRAFICOS = {}


def linha_meta(fig, valor, texto, eixo='y'):
    if eixo == 'y':
        fig.add_hline(y=valor, line=dict(color=CORES['texto'], width=1))
        fig.add_annotation(x=0, xref='paper', y=valor, text=texto, showarrow=False, xanchor='left', yanchor='bottom',
                           font=dict(size=11, color=CORES['texto']))
    else:
        fig.add_vline(x=valor, line=dict(color=CORES['texto'], width=1))
        fig.add_annotation(x=valor, y=1, yref='paper', text=texto, showarrow=False, yanchor='bottom',
                           font=dict(size=11, color=CORES['texto']))


# ---------- Aba 1 · Visão Geral ----------
rotulos = tf_tg_mensal['rotulo'].iloc[-12:].tolist()
for chave, serie, meta, casas, titulo in [('tf_12m', tf_12, METAS['tf'], 2, 'TF'), ('tg_12m', tg_12, METAS['tg'], 0, 'TG')]:
    fig = go.Figure(go.Scatter(
        x=rotulos, y=serie, mode='lines+markers', line=dict(color=CORES['azul'], width=2.5),
        marker=dict(size=6, color=CORES['azul']), fill='tozeroy', fillcolor='rgba(43,120,214,0.08)',
        hovertemplate=f'%{{x}}<br>{titulo} 12 meses: %{{y:,.{casas}f}}<extra></extra>'))
    linha_meta(fig, meta, f'Meta {num(meta)}')
    fig.add_annotation(x=rotulos[-1], y=serie.iloc[-1], text=f'<b>{num(serie.iloc[-1], casas)}</b>', showarrow=False,
                       xanchor='left', xshift=8, font=dict(size=12))
    fig.update_layout(height=300, showlegend=False, margin=dict(r=50),
                      yaxis=dict(range=[0, max(serie.max(), meta) * 1.25], tickformat=',.0f'))
    GRAFICOS[chave] = fig

ult12 = tf_tg_mensal.iloc[-12:]
fig = go.Figure([
    go.Bar(x=rotulos, y=ult12['acidentes_com_afastamento'], name='Com afastamento', marker_color=CORES['vermelho'],
           hovertemplate='%{x}<br>Com afastamento: %{y}<extra></extra>'),
    go.Bar(x=rotulos, y=ult12['acidentes_sem_afastamento'], name='Sem afastamento', marker_color=CORES['cinza'],
           hovertemplate='%{x}<br>Sem afastamento: %{y}<extra></extra>'),
])
fig.update_layout(barmode='stack', height=300, legend=dict(traceorder='normal'), yaxis=dict(dtick=1 if ult12[['acidentes_com_afastamento', 'acidentes_sem_afastamento']].sum(axis=1).max() <= 8 else None))
GRAFICOS['acidentes_mes'] = fig

fig = go.Figure(go.Bar(
    y=TF_POR_SETOR.index, x=TF_POR_SETOR.values, orientation='h',
    marker_color=[CORES['vermelho'] if v > METAS['tf'] else CORES['azul'] for v in TF_POR_SETOR.values],
    text=[num(v, 2) for v in TF_POR_SETOR.values], textposition='outside', cliponaxis=False,
    hovertemplate='%{y}<br>TF 12 meses: %{x:,.2f}<extra></extra>'))
linha_meta(fig, METAS['tf'], f'Meta {num(METAS["tf"])}', eixo='x')
fig.update_layout(height=300, showlegend=False, margin=dict(r=50, t=30),
                  xaxis=dict(range=[0, max(TF_POR_SETOR.max(), METAS['tf']) * 1.2], showgrid=True, gridcolor=CORES['grade']),
                  yaxis=dict(showgrid=False))
GRAFICOS['tf_setor'] = fig

# ---------- Aba 2 · Riscos e Brigada ----------
ROTULO_PROB = {1: '1 · Rara', 2: '2 · Improvável', 3: '3 · Possível', 4: '4 · Provável', 5: '5 · Quase certa'}
ROTULO_SEV = {1: '1 · Insignificante', 2: '2 · Menor', 3: '3 · Moderada', 4: '4 · Maior', 5: '5 · Catastrófica'}
ORDEM_GRAU = ['Baixo', 'Moderado', 'Alto', 'Crítico']
ESCALA_GRAU = [[0, COR_GRAU['Baixo']], [0.25, COR_GRAU['Baixo']], [0.25, COR_GRAU['Moderado']], [0.5, COR_GRAU['Moderado']],
               [0.5, COR_GRAU['Alto']], [0.75, COR_GRAU['Alto']], [0.75, COR_GRAU['Crítico']], [1, COR_GRAU['Crítico']]]


def legenda_grau(fig):
    for grau in ORDEM_GRAU:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=grau, hoverinfo='skip',
                                 marker=dict(symbol='square', size=12, color=COR_GRAU[grau])))


z, texto, dica = [], [], []
for p in range(1, 6):
    z.append([ORDEM_GRAU.index(grau_risco(p * s)) for s in range(1, 6)])
    linha_txt, linha_dica = [], []
    for s in range(1, 6):
        celula = riscos_nr1[(riscos_nr1['probabilidade'] == p) & (riscos_nr1['severidade'] == s)]
        linha_txt.append(f'<b>{len(celula)}</b>' if len(celula) else '')
        lista = '<br>'.join(f'{r.risco_id} · {r.perigo} ({r.setor})' for r in celula.itertuples()) or 'Nenhum perigo'
        linha_dica.append(f'<b>Nível {p * s} · {grau_risco(p * s)}</b><br>{lista}')
    texto.append(linha_txt)
    dica.append(linha_dica)
fig = go.Figure(go.Heatmap(
    z=z, x=list(ROTULO_SEV.values()), y=list(ROTULO_PROB.values()), text=texto, texttemplate='%{text}',
    textfont=dict(size=16, color=CORES['texto']), customdata=dica, hovertemplate='%{customdata}<extra></extra>',
    colorscale=ESCALA_GRAU, zmin=0, zmax=3, showscale=False, xgap=3, ygap=3))
legenda_grau(fig)
fig.update_layout(height=450, xaxis=dict(title='Severidade', side='bottom', showline=False), yaxis=dict(title='Probabilidade', showgrid=False),
                  legend=dict(x=0.5, xanchor='center', y=-0.2, yanchor='top'), margin=dict(t=20))
GRAFICOS['matriz'] = fig

maior_nivel = riscos_nr1.pivot_table(index='setor', columns='categoria', values='nivel', aggfunc='max').reindex(index=list(SETORES), columns=CATEGORIAS_RISCO)
fig = go.Figure(go.Heatmap(
    z=maior_nivel.map(lambda v: ORDEM_GRAU.index(grau_risco(v))).values, x=CATEGORIAS_RISCO, y=list(SETORES),
    text=maior_nivel.values, texttemplate='%{text}', textfont=dict(size=13, color=CORES['texto']),
    hovertemplate='%{y} · %{x}<br>Maior nível: %{text}<extra></extra>',
    colorscale=ESCALA_GRAU, zmin=0, zmax=3, showscale=False, xgap=3, ygap=3))
legenda_grau(fig)
fig.update_layout(height=450, yaxis=dict(autorange='reversed', showgrid=False), xaxis=dict(showline=False),
                  legend=dict(x=0.5, xanchor='center', y=-0.12, yanchor='top'), margin=dict(t=20))
GRAFICOS['setor_categoria'] = fig

grade_brigada = cobertura.pivot(index='predio', columns='turno', values='validos').reindex(index=PREDIOS, columns=TURNOS)
total_brigada = cobertura.pivot(index='predio', columns='turno', values='brigadistas').reindex(index=PREDIOS, columns=TURNOS)
nivel_brigada = grade_brigada.map(lambda v: 0 if v < BRIGADISTAS_MINIMOS else (1 if v == BRIGADISTAS_MINIMOS else 2))
fig = go.Figure(go.Heatmap(
    z=nivel_brigada.values, x=TURNOS, y=PREDIOS,
    text=[[f'<b>{grade_brigada.iloc[i, j]}</b> válidos<br>de {total_brigada.iloc[i, j]}' for j in range(3)] for i in range(3)],
    texttemplate='%{text}', textfont=dict(size=13, color=CORES['texto']),
    hovertemplate='%{y} · %{x}<br>%{text}<extra></extra>',
    colorscale=[[0, COR_GRAU['Crítico']], [0.33, COR_GRAU['Crítico']], [0.33, COR_GRAU['Moderado']], [0.66, COR_GRAU['Moderado']],
                [0.66, COR_GRAU['Baixo']], [1, COR_GRAU['Baixo']]],
    zmin=0, zmax=2, showscale=False, xgap=3, ygap=3))
for nome, cor in [(f'Abaixo de {BRIGADISTAS_MINIMOS}', COR_GRAU['Crítico']), (f'Exatamente {BRIGADISTAS_MINIMOS}', COR_GRAU['Moderado']),
                  (f'Acima de {BRIGADISTAS_MINIMOS}', COR_GRAU['Baixo'])]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=nome, hoverinfo='skip', marker=dict(symbol='square', size=12, color=cor)))
fig.update_layout(height=340, yaxis=dict(autorange='reversed', showgrid=False), xaxis=dict(showline=False),
                  legend=dict(y=-0.14, yanchor='top'), margin=dict(t=10, b=10))
GRAFICOS['brigada'] = fig

por_tipo = (equipamentos.groupby(['tipo', 'situacao']).size().unstack(fill_value=0)
            .reindex(columns=FAIXAS, fill_value=0).reindex(list(TIPOS_EQUIPAMENTO)[::-1]))
fig = go.Figure([go.Bar(y=por_tipo.index, x=por_tipo[f], name=f, orientation='h', marker_color=COR_FAIXA[f],
                        hovertemplate='%{y}<br>' + f + ': %{x}<extra></extra>') for f in FAIXAS])
fig.update_layout(barmode='stack', height=380, xaxis=dict(title='Itens', showgrid=True, gridcolor=CORES['grade']), yaxis=dict(showgrid=False),
                  legend=dict(y=-0.18, yanchor='top', traceorder='normal'), margin=dict(t=10))
GRAFICOS['equipamentos'] = fig

# ---------- Aba 3 · Governança ----------
docs = documentos_legais.sort_values('dias_para_vencer', ascending=False)
fig = go.Figure()
for faixa in FAIXAS:
    parte = docs[docs['situacao'] == faixa]
    fig.add_trace(go.Bar(
        y=parte['documento'], x=(parte['vencimento'] - parte['emissao']).dt.total_seconds() * 1000,
        base=parte['emissao'].dt.strftime('%Y-%m-%d'), orientation='h', name=faixa, marker_color=COR_FAIXA[faixa],
        customdata=np.stack([parte['emissao'].map(data_br), parte['vencimento'].map(data_br), parte['dias_para_vencer'].map(texto_prazo)], axis=-1)
        if len(parte) else None,
        hovertemplate='<b>%{y}</b><br>Emissão: %{customdata[0]}<br>Vencimento: %{customdata[1]}<br>%{customdata[2]}<extra></extra>',
        width=0.55))
for linha in docs.itertuples():
    fig.add_annotation(x=linha.vencimento, y=linha.documento, text=texto_prazo(linha.dias_para_vencer), showarrow=False,
                       xanchor='left', xshift=8, font=dict(size=12, color=CORES['texto_2'] if linha.situacao == 'Em dia' else CORES['vermelho']))
inicio = (docs['emissao'].min() - pd.DateOffset(months=2)).to_period('M').start_time
fim = docs['vencimento'].max() + pd.DateOffset(months=8)
marcos = [m for m in pd.date_range(inicio, fim, freq='MS') if m.month in (1, 7)]
fig.add_vline(x=DATA_REFERENCIA.strftime('%Y-%m-%d'), line=dict(color=CORES['texto'], width=1.5))
fig.add_annotation(x=DATA_REFERENCIA.strftime('%Y-%m-%d'), y=1, yref='paper', text='<b>Hoje</b>', showarrow=False, yanchor='bottom')
fig.update_layout(height=60 * len(docs) + 120, barmode='overlay',
                  xaxis=dict(type='date', range=[inicio, fim], tickvals=marcos, ticktext=[f'{MESES_PT[m.month - 1]}/{str(m.year)[2:]}' for m in marcos],
                             showgrid=True, gridcolor=CORES['grade']),
                  yaxis=dict(categoryorder='array', categoryarray=docs['documento'].tolist()[::-1], showgrid=False),
                  legend=dict(y=-0.12, yanchor='top'), margin=dict(t=30))
GRAFICOS['documentos'] = fig

# ---------- Aba 4 · Percepção ----------
ordem_dim = FAVORAVEL.sort_values().index.tolist()     # pior embaixo
dados_ult = ultimo.loc[ordem_dim]
fig = go.Figure()
lados = [('Neutro', -0.5, True), ('Discordo', -1, True), ('Discordo totalmente', -1, True),
         ('Neutro', 0.5, False), ('Concordo', 1, True), ('Concordo totalmente', 1, True)]
for nome, fator, legenda in lados:
    valores = dados_ult[nome] * abs(fator)
    fig.add_trace(go.Bar(
        y=ordem_dim, x=valores * np.sign(fator), orientation='h', name=nome, marker_color=COR_LIKERT[nome],
        showlegend=legenda, legendgroup=nome, legendrank=list(COR_LIKERT).index(nome), customdata=dados_ult[nome],
        hovertemplate='%{y}<br>' + nome + ': %{customdata:.1f}%<extra></extra>'))
for dim in ordem_dim:
    fav = dados_ult.loc[dim, 'favoravel']
    fig.add_annotation(x=dados_ult.loc[dim, 'Concordo'] + dados_ult.loc[dim, 'Concordo totalmente'] + dados_ult.loc[dim, 'Neutro'] / 2,
                       y=dim, text=f'<b>{pct(fav)}</b>', showarrow=False, xanchor='left', xshift=6,
                       font=dict(size=12, color=CORES['texto'] if fav >= METAS['favoravel'] else CORES['vermelho']))
fig.update_layout(barmode='relative', height=440, bargap=0.35,
                  xaxis=dict(range=[-62, 105], tickvals=list(range(-60, 101, 20)), ticktext=[f'{abs(v)}%' for v in range(-60, 101, 20)],
                             zeroline=True, zerolinecolor=CORES['eixo'], showgrid=True, gridcolor=CORES['grade']),
                  yaxis=dict(showgrid=False), legend=dict(traceorder='normal'))
GRAFICOS['likert'] = fig

participacao_ciclos = pesquisa_resumo[pesquisa_resumo['dimensao'].str.startswith('Geral')]['participacao'].tolist()
fig = go.Figure(go.Bar(
    x=CICLOS, y=participacao_ciclos, marker_color=[CORES['cinza']] * (len(CICLOS) - 1) + [CORES['azul']],
    text=[f'<b>{pct(v)}</b>' for v in participacao_ciclos], textposition='inside', insidetextanchor='end',
    textfont=dict(color=[CORES['texto']] * (len(CICLOS) - 1) + ['#FFFFFF'], size=13),
    hovertemplate='%{x}<br>Participação: %{y:.1f}%<extra></extra>'))
linha_meta(fig, METAS['participacao'], f'Meta {METAS["participacao"]}%')
fig.update_layout(height=440, showlegend=False, bargap=0.45, yaxis=dict(range=[0, 100], ticksuffix='%', dtick=20))
GRAFICOS['participacao'] = fig

fig = go.Figure()
for i, dim in enumerate(DIMENSOES):
    serie = pesquisa_resumo[pesquisa_resumo['dimensao'] == dim]['favoravel']
    fig.add_trace(go.Scatter(x=CICLOS, y=serie, mode='lines+markers', name=dim, line=dict(width=2.5, color=CORES_SERIES[i]),
                             marker=dict(size=7), hovertemplate=dim + '<br>%{x}: %{y:.1f}%<extra></extra>'))
linha_meta(fig, METAS['favoravel'], f'Meta {METAS["favoravel"]}%')
fig.update_layout(height=400, yaxis=dict(range=[20, 100], ticksuffix='%'), legend=dict(y=-0.15, yanchor='top'), margin=dict(t=20))
GRAFICOS['evolucao'] = fig

do_ultimo = pesquisa_respostas[pesquisa_respostas['ciclo'] == CICLOS[-1]]
mapa = (do_ultimo.assign(fav=do_ultimo['nota'] >= 4).pivot_table(index='setor', columns='dimensao', values='fav', aggfunc='mean') * 100)
mapa = mapa.reindex(index=list(SETORES), columns=DIMENSOES)
fig = go.Figure(go.Heatmap(
    z=mapa.values, x=['<br>'.join(textwrap.wrap(d, 16)) for d in DIMENSOES], y=list(SETORES),
    text=[[pct(v) for v in linha] for linha in mapa.values], texttemplate='%{text}', textfont=dict(size=12, color=CORES['texto']),
    hovertemplate='%{y}<br>%{x}<br>Favorável: %{text}<extra></extra>',
    colorscale=[[0, '#F0A8A8'], [0.5, '#FCEBB6'], [1, '#BFE3C3']], zmin=METAS['favoravel'] - 30, zmax=METAS['favoravel'] + 20,
    zmid=METAS['favoravel'], xgap=3, ygap=3, colorbar=dict(ticksuffix='%', thickness=12, len=0.8, outlinewidth=0)))
fig.update_layout(height=420, yaxis=dict(autorange='reversed', showgrid=False), xaxis=dict(side='top', showline=False, tickangle=0), margin=dict(t=70))
GRAFICOS['mapa_setor'] = fig

# ---------- Aba 5 · Selos e Certificações ----------
ating = selos_certificacoes.sort_values('atingimento', ascending=False)
simbolo = {'OK': '✓', 'Atenção': '!', 'Alerta Vermelho': '✕'}


def valor_ind(linha):
    return pct(linha['atual'], 1) if linha['unidade'] == '%' else num(linha['atual'], 2 if 'Frequência' in linha['indicador'] else 0)


def meta_ind(linha):
    return f'{"≤ " if linha["sentido"] == "<=" else ""}{num(linha["meta"])}{linha["unidade"]}'


fig = go.Figure(go.Bar(
    y=[f'{i} · {s}' for i, s in zip(ating['indicador'], ating['selo'])], x=ating['atingimento'].clip(upper=150), orientation='h',
    marker_color=[COR_SITUACAO[s] for s in ating['situacao']],
    text=[f'{simbolo[l["situacao"]]} {valor_ind(l)} de meta {meta_ind(l)}' for _, l in ating.iterrows()],
    textposition='outside', cliponaxis=False, textfont=dict(size=11, color=CORES['texto_2']),
    customdata=ating['situacao'], hovertemplate='%{y}<br>Atingimento: %{x:.1f}% da meta<br>%{customdata}<extra></extra>'))
linha_meta(fig, 100, '<b>Meta</b>', eixo='x')
fig.update_layout(height=34 * len(ating) + 90, showlegend=False, bargap=0.35, margin=dict(r=180, t=30),
                  xaxis=dict(range=[0, 155], ticksuffix='%', showgrid=True, gridcolor=CORES['grade']), yaxis=dict(showgrid=False))
GRAFICOS['atingimento'] = fig

print(f'{len(GRAFICOS)} gráficos criados: {", ".join(GRAFICOS)}')
GRAFICOS['matriz']

## Etapa 14 · Montar o painel em HTML

Junta cards, tabelas e gráficos em um único arquivo HTML com cinco abas. A biblioteca Plotly vai **embutida** no arquivo, por isso o painel funciona sem internet. Ao imprimir (Ctrl + P), as abas saem em sequência, como relatório.

In [ ]:
def grafico(nome):
    return pio.to_html(GRAFICOS[nome], full_html=False, include_plotlyjs=False, config=CONFIG_GRAFICO, div_id=f'g-{nome}')


def secao(titulo):
    return f'<h2 class="secao">{titulo}</h2>'


# ---------- Aba 1 ----------
aba1 = (leitura(LEITURA[1]) + f'<div class="kpis">{cards_visao}</div>'
        + cartao('Alertas que pedem ação', 'Críticos primeiro · clique para ir à aba correspondente', lista_alertas, 'cartao-alerta')
        + '<div class="grade">'
        + cartao('Taxa de Frequência', 'Acumulado móvel de 12 meses · acidentes com afastamento por milhão de HHT', grafico('tf_12m'))
        + cartao('Taxa de Gravidade', 'Acumulado móvel de 12 meses · dias perdidos e debitados por milhão de HHT', grafico('tg_12m'))
        + cartao('Acidentes por mês', 'Com e sem afastamento · últimos 12 meses', grafico('acidentes_mes'))
        + cartao('Taxa de Frequência por setor', 'Acumulado de 12 meses · vermelho acima da meta', grafico('tf_setor'))
        + '</div>')

# ---------- Aba 2 ----------
tab_riscos = riscos_sem_plano.assign(
    status=riscos_sem_plano['status_plano'].map(selo_html), prazo=riscos_sem_plano['prazo_plano'].map(data_br))
validade_brig = brigada[brigada['situacao'] != 'Em dia'].sort_values('dias_para_vencer').assign(
    venc=lambda d: d['vencimento_reciclagem'].map(data_br), sit=lambda d: d['situacao'].map(selo_html),
    exame=lambda d: d['exame_especifico'].map(selo_html))
validade_eq = equipamentos[equipamentos['situacao'] != 'Em dia'].sort_values('dias_para_vencer').assign(
    val=lambda d: d['validade'].map(data_br), sit=lambda d: d['situacao'].map(selo_html))
aba2 = (leitura(LEITURA[2]) + secao('Mapeamento de riscos · NR-1') + '<div class="grade">'
        + cartao('Matriz de risco 5×5', 'Número de perigos em cada combinação · passe o mouse para ver quais são', grafico('matriz'))
        + cartao('Maior nível de risco por setor e categoria', 'Número = nível (probabilidade × severidade)', grafico('setor_categoria'))
        + '</div>'
        + cartao('Riscos altos e críticos sem plano de ação em dia', 'Prioridade de tratamento no PGR',
                 tabela(tab_riscos, {'risco_id': 'ID', 'setor': 'Setor', 'categoria': 'Categoria', 'perigo': 'Perigo', 'nivel': 'Nível',
                                     'grau': 'Grau', 'responsavel': 'Responsável', 'prazo': 'Prazo', 'status': 'Plano de ação'}, numericas=('nivel',)))
        + secao('Brigada de incêndio · NR-23') + '<div class="grade">'
        + cartao('Cobertura da brigada por prédio e turno', f'Brigadistas com reciclagem válida · mínimo de {BRIGADISTAS_MINIMOS} por prédio e turno', grafico('brigada'))
        + cartao('Brigadistas com reciclagem vencida ou a vencer', 'Reciclagem anual · vencidos e próximos 60 dias',
                 tabela(validade_brig, {'colaborador_id': 'Colaborador', 'predio': 'Prédio', 'turno': 'Turno',
                                        'venc': 'Vencimento', 'exame': 'Exame', 'sit': 'Situação'}), 'rolagem')
        + '</div>' + secao('Equipamentos de segurança') + '<div class="grade">'
        + cartao('Validade dos equipamentos por tipo', 'Extintores, hidrantes, iluminação de emergência, primeiros socorros e EPIs críticos', grafico('equipamentos'))
        + cartao('Equipamentos vencidos ou a vencer', 'Itens vencidos e com vencimento em até 60 dias',
                 tabela(validade_eq, {'equipamento_id': 'Item', 'tipo': 'Tipo', 'predio': 'Prédio', 'val': 'Validade',
                                      'dias_para_vencer': 'Dias', 'sit': 'Situação'}, numericas=('dias_para_vencer',)), 'rolagem')
        + '</div>')

# ---------- Aba 3 ----------
tab_docs = documentos_legais.assign(em=documentos_legais['emissao'].map(data_br), venc=documentos_legais['vencimento'].map(data_br),
                                    sit=documentos_legais['situacao'].map(selo_html))
colunas_quadro = []
for etapa in ETAPAS_FLUXO:
    da_etapa = pops_pae[pops_pae['etapa'] == etapa].sort_values('dias_na_etapa', ascending=False)
    cartoes = ''.join(
        f'<div class="fluxo {"parado" if l.parado else ""}"><b>{l.codigo}</b><span>{html.escape(l.titulo)}</span>'
        f'<small>{"vigente desde " + data_br(l.ultima_revisao) if etapa == "Vigente" else str(l.dias_na_etapa) + " dias nesta etapa"}'
        f'{" · parado" if l.parado else ""}</small></div>' for l in da_etapa.itertuples()) or '<p class="vazio">—</p>'
    colunas_quadro.append(f'<div class="coluna {"col-vigente" if etapa == "Vigente" else ""} {"col-vencido" if etapa == "Vencido" else ""}">'
                          f'<h4>{etapa}<span>{len(da_etapa)}</span></h4>{cartoes}</div>')
aba3 = (leitura(LEITURA[3])
        + cartao('Vigência dos documentos legais', 'Cada barra vai da emissão ao vencimento · a linha preta marca hoje', grafico('documentos'))
        + cartao('Documentos legais obrigatórios', '',
                 tabela(tab_docs, {'documento': 'Documento', 'norma': 'Norma', 'responsavel': 'Responsável', 'em': 'Emissão',
                                   'venc': 'Vencimento', 'dias_para_vencer': 'Dias', 'sit': 'Situação'}, numericas=('dias_para_vencer',)))
        + cartao('Fluxo de revisão dos POPs e do PAE', f'Etapa atual de cada procedimento · em vermelho, parado há mais de {DIAS_FLUXO_PARADO} dias',
                 f'<div class="quadro">{"".join(colunas_quadro)}</div>'))

# ---------- Aba 4 ----------
dim_forte, dim_fragil = FAVORAVEL.idxmax(), FAVORAVEL.idxmin()


def mini(titulo, valor, situacao, detalhe):
    pill = selo_html(situacao) if situacao else ''
    return f'<div class="mini-card"><span class="kpi-titulo">{titulo}</span><div class="kpi-valor">{valor}</div>{pill}<p>{detalhe}</p></div>'


resumo_ciclo = ''.join([
    mini(f'Satisfação em SST · {CICLOS[-1]}', pct(SATISFACAO), situacao_de('Satisfação geral em SST'),
         f'meta {METAS["favoravel"]}% · {variacao_html(SATISFACAO - SATISFACAO_ANTERIOR, False, " p.p. vs ciclo anterior", 1)}'),
    mini('Participação', pct(PARTICIPACAO), situacao_de('Participação na pesquisa'), f'{PARTICIPANTES} de {EFETIVO} colaboradores'),
    mini('Dimensão mais forte', pct(FAVORAVEL.max()), None, dim_forte),
    mini('Dimensão mais frágil', pct(FAVORAVEL.min()), avaliar(FAVORAVEL.min(), METAS['favoravel'], '>=')[1], dim_fragil),
])
aba4 = (leitura(LEITURA[4]) + cartao('Resumo do ciclo', '', f'<div class="minis">{resumo_ciclo}</div>')
        + '<div class="grade grade-larga">'
        + cartao(f'Percepção de segurança – {CICLOS[-1]}', 'À direita do zero, concordância; à esquerda, discordância · número = % favorável (vermelho abaixo da meta)', grafico('likert'))
        + cartao('Participação na pesquisa', 'Percentual de colaboradores que responderam em cada ciclo', grafico('participacao'))
        + '</div><div class="grade">'
        + cartao('Evolução do % favorável por dimensão', 'Respostas 4 e 5 em cada ciclo trimestral', grafico('evolucao'))
        + cartao(f'Mapa por setor e dimensão – {CICLOS[-1]}', '% favorável · verde acima da meta, vermelho abaixo', grafico('mapa_setor'))
        + '</div>')

# ---------- Aba 5 ----------
cards_selos = ''
for linha in resumo_selos.itertuples():
    cards_selos += (f'<div class="kpi selo-card"><div class="kpi-topo"><span class="kpi-titulo">{linha.selo}</span>{selo_html(linha.situacao)}</div>'
                    f'<div class="kpi-valor">{linha.ok}<small> de {linha.indicadores}</small></div><div class="kpi-unidade">indicadores na meta</div>'
                    f'<div class="barra"><div class="barra-cheia" style="width:{100 * linha.ok / linha.indicadores:.0f}%"></div></div>'
                    f'<div class="kpi-rodape"><span>{linha.alerta} em alerta vermelho · {linha.atencao} em atenção</span></div></div>')
tab_alerta = em_alerta.sort_values('distancia_meta', ascending=False).assign(
    at=lambda d: d.apply(valor_ind, axis=1), mt=lambda d: d.apply(meta_ind, axis=1))
tab_todos = selos_certificacoes.assign(at=lambda d: d.apply(valor_ind, axis=1), mt=lambda d: d.apply(meta_ind, axis=1),
                                       sit=lambda d: d['situacao'].map(selo_html))
aba5 = (leitura(LEITURA[5]) + f'<div class="kpis tres">{cards_selos}</div>'
        + cartao('Tabela de Alerta Vermelho', 'Indicadores que puxam a nota para baixo e ameaçam a conquista ou a renovação dos selos',
                 tabela(tab_alerta, {'indicador': 'Indicador', 'selo': 'Selo', 'criterio': 'Critério', 'at': 'Atual', 'mt': 'Meta',
                                     'plano_mitigacao': 'Plano de mitigação sugerido'}, numericas=('at', 'mt'), quebra=('plano_mitigacao',)), 'cartao-alerta')
        + cartao('Atingimento da meta por indicador', '100% = meta cumprida · verde OK, amarelo atenção, vermelho alerta', grafico('atingimento'))
        + cartao('Todos os indicadores', 'Situação de cada indicador por selo e critério',
                 tabela(tab_todos, {'indicador': 'Indicador', 'selo': 'Selo', 'criterio': 'Critério', 'at': 'Atual', 'mt': 'Meta',
                                    'sit': 'Situação'}, numericas=('at', 'mt'))))

ABAS = [('Visão Geral', aba1), ('Riscos e Brigada', aba2), ('Governança', aba3), ('Percepção', aba4), ('Selos e Certificações', aba5)]
navegacao = ''.join(
    f'<button type="button" class="{"ativa" if i == 1 else ""}" data-aba="{i}" onclick="abrirAba({i})"><span class="num">{i}</span>{nome}'
    f'{f"<span class=badge>{ALERTAS_POR_ABA[i]}</span>" if ALERTAS_POR_ABA[i] else ""}</button>'
    for i, (nome, _) in enumerate(ABAS, start=1))
conteudo_abas = ''.join(f'<section class="aba {"ativa" if i == 1 else ""}" id="aba-{i}"><h2 class="titulo-impressao">{i}. {nome}</h2>{corpo}</section>'
                        for i, (nome, corpo) in enumerate(ABAS, start=1))

CSS = '''
:root{--marinho:#0F3466;--azul:#2B78D6;--texto:#252423;--texto-2:#605E5C;--borda:#E6E4DE;--fundo:#F3F2EE;--cartao:#FCFCFB;--vermelho:#D13438}
*{box-sizing:border-box}
body{margin:0;font-family:Inter,"Segoe UI",Roboto,"Helvetica Neue",Arial,sans-serif;background:var(--fundo);color:var(--texto);font-size:14px;line-height:1.45}
header{background:var(--marinho);color:#fff;padding:22px 32px 20px;display:flex;align-items:center;justify-content:space-between;gap:16px;flex-wrap:wrap}
header h1{margin:0;font-size:24px;font-weight:600}
header p{margin:4px 0 0;font-size:13px;opacity:.82}
.demo{border:1px solid rgba(255,255,255,.45);border-radius:999px;padding:5px 12px;font-size:12px;white-space:nowrap}
nav{position:sticky;top:0;z-index:20;display:flex;gap:2px;overflow-x:auto;background:#fff;border-bottom:1px solid var(--borda);padding:0 20px}
nav button{display:inline-flex;align-items:center;gap:8px;border:0;background:none;padding:14px 14px 12px;font:inherit;font-size:14px;color:var(--texto-2);
  border-bottom:3px solid transparent;cursor:pointer;white-space:nowrap}
nav button .num{display:grid;place-items:center;width:22px;height:22px;border-radius:50%;background:#EDEBE6;font-size:12px;color:var(--texto-2)}
nav button.ativa{color:var(--marinho);font-weight:600;border-bottom-color:var(--azul)}
nav button.ativa .num{background:var(--azul);color:#fff}
nav button:hover{color:var(--marinho)}
.badge{background:var(--vermelho);color:#fff;border-radius:999px;font-size:11px;font-weight:700;padding:0 7px;line-height:18px}
main{max-width:1440px;margin:0 auto;padding:24px 32px 8px}
.aba{display:none}.aba.ativa{display:block}
.titulo-impressao{display:none}
.leitura{display:flex;gap:16px;align-items:flex-start;background:#fff;border:1px solid var(--borda);border-left:4px solid var(--azul);border-radius:6px;padding:16px 20px;margin-bottom:20px}
.leitura-rotulo{flex:none;font-size:11px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:var(--azul);padding-top:3px}
.leitura p{margin:0;font-size:15px}
h2.secao{font-size:14px;font-weight:700;letter-spacing:.06em;text-transform:uppercase;color:var(--marinho);margin:28px 0 12px}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:16px;margin-bottom:20px}
.kpis.tres{grid-template-columns:repeat(3,minmax(0,1fr))}
.kpi{background:var(--cartao);border:1px solid var(--borda);border-radius:8px;padding:18px 18px 14px;display:flex;flex-direction:column;gap:8px}
.kpi-topo{display:flex;justify-content:space-between;align-items:center;gap:8px}
.kpi-titulo{font-size:13px;color:var(--texto-2)}
.kpi-meio{display:flex;justify-content:space-between;align-items:flex-end;gap:8px;flex-wrap:wrap}
.kpi-valor{font-size:36px;font-weight:600;line-height:1.1}
.kpi-valor small{font-size:16px;font-weight:400;color:var(--texto-2)}
.kpi-unidade{font-size:12px;color:var(--texto-2)}
.kpi-rodape{display:flex;justify-content:space-between;gap:8px;flex-wrap:wrap;border-top:1px solid var(--borda);padding-top:10px;font-size:12px;color:var(--texto-2)}
.var{font-weight:600}.var.boa{color:#1E7A30}.var.ruim{color:var(--vermelho)}.var.neutra{font-weight:400;color:var(--texto-2)}
.mini{flex:none}
.barra{position:relative;width:100%;height:8px;background:#E3EEFB;border-radius:4px;margin:6px 0 2px}
.barra-cheia{height:100%;background:var(--azul);border-radius:4px}
.barra-meta{position:absolute;top:-4px;width:2px;height:16px;background:var(--texto)}
.pill{display:inline-flex;align-items:center;gap:6px;padding:2px 10px 2px 3px;border-radius:999px;font-size:12px;font-weight:600;border:1px solid;white-space:nowrap;width:fit-content}
.pill .ic,.pill-ic{display:grid;place-items:center;width:16px;height:16px;border-radius:50%;color:#fff;font-size:10px;font-weight:700;flex:none}
.pill.ok{background:#E8F5E9;border-color:#A9D5AC;color:#1E5E2A}.pill.ok .ic{background:#2E8B3E}
.pill.atencao{background:#FFF6D6;border-color:#F3D27A;color:#6B4E00}.pill.atencao .ic{background:#F2B200}
.pill.laranja{background:#FDEDE4;border-color:#F4C0A6;color:#8A3B12}.pill.laranja .ic{background:#E8703A}
.pill.alerta{background:#FDE7E7;border-color:#F1B2B2;color:#8A1F22}.pill.alerta .ic{background:var(--vermelho)}
.pill-ic{width:22px;height:22px;border-radius:4px;font-size:12px}
.pill-ic.alerta{background:var(--vermelho)}.pill-ic.atencao{background:#F2B200;color:#252423}
.cartao{background:var(--cartao);border:1px solid var(--borda);border-radius:8px;padding:18px 18px 12px;margin-bottom:20px;min-width:0}
.cartao h3{margin:0;font-size:16px;font-weight:600}
.cartao .sub{margin:3px 0 8px;font-size:12px;color:var(--texto-2)}
.cartao-alerta{border-top:4px solid var(--vermelho)}
.alertas{list-style:none;margin:6px 0 0;padding:0;display:grid;grid-template-columns:repeat(2,minmax(0,1fr));column-gap:28px}
.alertas li{display:flex;align-items:center;gap:12px;padding:11px 2px;border-bottom:1px solid var(--borda)}
.alertas .txt{flex:1}
.alertas a{color:var(--azul);text-decoration:none;font-size:13px;white-space:nowrap}
.alertas a:hover{text-decoration:underline}
.grade{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:0 20px;align-items:start}
.grade-larga{grid-template-columns:minmax(0,1.6fr) minmax(0,1fr)}
.tabela{overflow-x:auto}
.rolagem .tabela{max-height:330px;overflow-y:auto}
table{width:100%;border-collapse:collapse;font-size:13px}
th{position:sticky;top:0;background:#F3F2EE;text-align:left;font-weight:600;padding:9px 10px;border-bottom:1px solid var(--borda);white-space:nowrap}
td{padding:9px 10px;border-bottom:1px solid var(--borda);vertical-align:middle;white-space:nowrap}
td.q{white-space:normal;min-width:220px}
th.n,td.n{text-align:right;white-space:nowrap}
td.vazio,p.vazio{color:var(--texto-2);text-align:center}
.minis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:14px;margin-top:8px}
.mini-card{border:1px solid var(--borda);border-radius:6px;padding:14px;display:flex;flex-direction:column;gap:6px;background:#fff}
.mini-card .kpi-valor{font-size:28px}
.mini-card p{margin:0;font-size:12px;color:var(--texto-2)}
.quadro{display:grid;grid-template-columns:repeat(6,minmax(150px,1fr));gap:10px;overflow-x:auto;margin-top:8px}
.coluna{background:#F3F2EE;border-radius:6px;padding:10px;min-height:120px}
.coluna h4{display:flex;justify-content:space-between;margin:0 0 8px;font-size:12px;font-weight:600;color:var(--texto-2);text-transform:uppercase;letter-spacing:.03em}
.coluna h4 span{background:#fff;border-radius:999px;padding:0 8px;color:var(--texto)}
.col-vigente{background:#E8F5E9}.col-vencido{background:#FDE7E7}
.fluxo{background:#fff;border:1px solid var(--borda);border-left:4px solid var(--azul);border-radius:4px;padding:8px 9px;margin-bottom:8px;display:flex;flex-direction:column;gap:2px;font-size:12px}
.fluxo span{color:var(--texto)}.fluxo small{color:var(--texto-2)}
.fluxo.parado{border-left-color:var(--vermelho)}.fluxo.parado small{color:var(--vermelho);font-weight:600}
.col-vigente .fluxo{border-left-color:#2E8B3E}
footer{text-align:center;font-size:12px;color:#8A8886;padding:12px 16px 28px}
@media (max-width:1100px){.kpis,.minis{grid-template-columns:repeat(2,minmax(0,1fr))}.kpis.tres{grid-template-columns:1fr}}
@media (max-width:860px){.grade,.grade-larga,.alertas{grid-template-columns:1fr}main{padding:16px}header{padding:18px 16px}}
@media (max-width:560px){.kpis,.minis{grid-template-columns:1fr}.leitura{flex-direction:column;gap:6px}}
@media print{
  nav,.demo{display:none}
  body{background:#fff}
  .aba{display:block!important;break-before:page}
  .aba:first-child{break-before:auto}
  .titulo-impressao{display:block;font-size:18px;color:var(--marinho);margin:0 0 12px}
  .cartao,.kpi,.leitura{break-inside:avoid}
  .rolagem .tabela{max-height:none}
}
'''

JS = '''
function redimensionar(raiz){ if(window.Plotly){ raiz.querySelectorAll('.js-plotly-plot').forEach(function(g){ Plotly.Plots.resize(g); }); } }
function abrirAba(n){
  document.querySelectorAll('.aba').forEach(function(a){ a.classList.toggle('ativa', a.id === 'aba-' + n); });
  document.querySelectorAll('nav button').forEach(function(b){ b.classList.toggle('ativa', b.dataset.aba == n); });
  if(history.replaceState){ history.replaceState(null, '', '#aba-' + n); }
  window.scrollTo(0, 0);
  setTimeout(function(){ redimensionar(document.getElementById('aba-' + n)); }, 30);
}
window.addEventListener('beforeprint', function(){ document.querySelectorAll('.aba').forEach(function(a){ a.style.display = 'block'; }); redimensionar(document); });
window.addEventListener('afterprint', function(){ document.querySelectorAll('.aba').forEach(function(a){ a.style.display = ''; }); redimensionar(document); });
function iniciar(){
  var m = location.hash.match(/^#aba-([1-5])$/);
  abrirAba(m ? +m[1] : 1);
}
if(document.readyState === 'complete'){ iniciar(); } else { window.addEventListener('load', iniciar); }
'''


def montar_painel(script_plotly):
    return f'''<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Painel de SST e Conformidade · versão Python</title>
<style>{CSS}</style>
{script_plotly}
</head>
<body>
<header>
  <div><h1>Saúde, Segurança do Trabalho e Conformidade</h1>
  <p>{EMPRESA} · Período: {PERIODO_TEXTO} · Atualizado em {data_br(DATA_REFERENCIA)}</p></div>
  <span class="demo">Dados fictícios para demonstração</span>
</header>
<nav aria-label="Abas do painel">{navegacao}</nav>
<main>{conteudo_abas}</main>
<footer>Painel gerado em Python com pandas, NumPy e Plotly · dados fictícios (semente {SEMENTE}) · versão 2 do projeto painel-sst-conformidade</footer>
<script>{JS}</script>
</body>
</html>'''


PAINEL_HTML = montar_painel(f'<script type="text/javascript">{get_plotlyjs()}</script>')
print(f'Painel montado: {len(PAINEL_HTML.encode("utf-8")) / 1e6:.1f} MB, com a biblioteca Plotly embutida.')

## Etapa 15 · Ver a prévia do painel no Colab

Mostra o painel aqui mesmo, em uma janela com rolagem e com as abas funcionando. Para deixar o notebook leve, a prévia carrega a biblioteca Plotly pela internet; o arquivo exportado na Etapa 16 traz a biblioteca embutida.

In [ ]:
versao_plotly = plotly.offline.get_plotlyjs_version()
previa = montar_painel(f'<script src="https://cdn.plot.ly/plotly-{versao_plotly}.min.js"></script>')
display(HTML(f'<iframe srcdoc="{html.escape(previa)}" title="Prévia do painel" '
             'style="width:100%;height:900px;border:1px solid #E6E4DE;border-radius:8px"></iframe>'))

## Etapa 16 · Exportar o Excel e baixar os arquivos

Salva o painel (`painel_sst.html`) e as doze tabelas geradas (`base_sst.xlsx`). No Colab, os dois arquivos são baixados automaticamente; se o navegador perguntar, permita vários downloads. Fora do Colab, eles ficam na mesma pasta do notebook.

In [ ]:
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

PASTA_SAIDA = Path('.')
ARQUIVO_PAINEL = PASTA_SAIDA / 'painel_sst.html'
ARQUIVO_BASE = PASTA_SAIDA / 'base_sst.xlsx'

ARQUIVO_PAINEL.write_text(PAINEL_HTML, encoding='utf-8')

PLANILHAS = {
    'colaboradores': colaboradores,
    'horas_trabalhadas': horas_trabalhadas,
    'acidentes': acidentes,
    'tf_tg_mensal': tf_tg_mensal,
    'riscos_nr1': riscos_nr1,
    'brigada': brigada,
    'equipamentos': equipamentos,
    'documentos_legais': documentos_legais,
    'pops_pae': pops_pae,
    'pesquisa_respostas': pesquisa_respostas,
    'pesquisa_resumo': pesquisa_resumo,
    'selos_certificacoes': selos_certificacoes,
}
cabecalho = PatternFill('solid', start_color='0F3466')
with pd.ExcelWriter(ARQUIVO_BASE, engine='openpyxl') as escritor:
    for nome, df in PLANILHAS.items():
        df.to_excel(escritor, sheet_name=nome, index=False)
        ws = escritor.sheets[nome]
        for celula in ws[1]:
            celula.font = Font(name='Arial', bold=True, color='FFFFFF')
            celula.fill = cabecalho
            celula.alignment = Alignment(vertical='center')
        for i, coluna in enumerate(df.columns, start=1):
            serie = df[coluna]
            if pd.api.types.is_datetime64_any_dtype(serie):
                for (celula,) in ws.iter_rows(min_row=2, min_col=i, max_col=i):
                    celula.number_format = 'DD/MM/YYYY'
                largura = 12
            else:
                largura = min(60, max(len(str(coluna)), serie.astype(str).str.len().max() if len(serie) else 0) + 2)
            ws.column_dimensions[get_column_letter(i)].width = max(10, largura)
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions

print(f'Painel: {ARQUIVO_PAINEL.resolve()} ({ARQUIVO_PAINEL.stat().st_size / 1e6:.1f} MB)')
print(f'Base:   {ARQUIVO_BASE.resolve()} ({len(PLANILHAS)} planilhas)')
for nome, df in PLANILHAS.items():
    print(f'  · {nome}: {len(df)} linhas')

if NO_COLAB:
    arquivos_colab.download(str(ARQUIVO_PAINEL))
    arquivos_colab.download(str(ARQUIVO_BASE))

## Etapa 17 · Salvar os arquivos no Google Drive (opcional)

Para guardar uma cópia no seu Google Drive, troque `SALVAR_NO_DRIVE` para `True` e execute esta célula. O Colab pede autorização para acessar o Drive, e os arquivos ficam na pasta `painel_sst`.

In [ ]:
SALVAR_NO_DRIVE = False

if SALVAR_NO_DRIVE and NO_COLAB:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    destino = Path('/content/drive/MyDrive/painel_sst')
    destino.mkdir(parents=True, exist_ok=True)
    for arquivo in (ARQUIVO_PAINEL, ARQUIVO_BASE):
        shutil.copy(arquivo, destino / arquivo.name)
    print(f'Arquivos salvos em {destino}')
elif SALVAR_NO_DRIVE:
    print('O Google Drive só está disponível no Colab.')
else:
    print('Cópia no Google Drive desativada. Para ativar, troque SALVAR_NO_DRIVE para True.')